In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# Pirate Pain – Time Series Classification with Conv2D + Correlation-based channel ordering
# ============================================================

# ==== SECTION 0: Install dependencies (Colab) ====
# !pip -q install pandas scikit-learn torch torchvision torchaudio kaggle --upgrade


import torch
import torch.nn as nn
import torch.nn.functional as F

class ResidualBlock2D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=(3,3), stride=1, dropout=0.0):
        super().__init__()
        padding = (kernel_size[0] // 2, kernel_size[1] // 2)

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size,
                               stride=stride, padding=padding, bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=kernel_size,
                               stride=1, padding=padding, bias=False)
        self.bn2   = nn.BatchNorm2d(out_channels)

        # Optional projection if in/out channels or stride differ
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

        self.dropout = nn.Dropout2d(dropout)

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = F.relu(out, inplace=True)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.dropout(out)

        out = out + identity
        out = F.relu(out, inplace=True)
        return out


class CNN2D_with_Static(nn.Module):
    def __init__(self, static_dim, num_classes):
        super().__init__()

        # Input x_dyn: (B, 1, T, C_dyn)

        # Stem
        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=(3,3), padding=(1,1), bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )

        # Residual stages
        self.layer1 = nn.Sequential(
            ResidualBlock2D(32, 32, kernel_size=(3,3), stride=1, dropout=0.1),
            ResidualBlock2D(32, 32, kernel_size=(3,3), stride=1, dropout=0.1),
        )
        # Downsample in both time and feature axes
        self.pool1 = nn.MaxPool2d(kernel_size=(2,2))   # (T/2, C/2)

        self.layer2 = nn.Sequential(
            ResidualBlock2D(32, 64, kernel_size=(3,3), stride=1, dropout=0.15),
            ResidualBlock2D(64, 64, kernel_size=(3,3), stride=1, dropout=0.15),
        )
        self.pool2 = nn.MaxPool2d(kernel_size=(2,2))   # (T/4, C/4)

        # Optional extra depth without pooling
        self.layer3 = nn.Sequential(
            ResidualBlock2D(64, 64, kernel_size=(3,3), stride=1, dropout=0.2),
        )

        # Adaptive pooling to fixed spatial size
        self.spatial_pool = nn.AdaptiveAvgPool2d((4, 4))  # (B, 64, 4, 4)
        self.cnn_out_dim = 64 * 4 * 4

        # MLP for static features
        self.static_fc = nn.Sequential(
            nn.Linear(static_dim, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
        )

        # Final classifier on concatenated features
        self.fc = nn.Sequential(
            nn.Linear(self.cnn_out_dim + 64, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x_dyn, x_stat):
        # x_dyn:  (B, 1, T, C_dyn)
        # x_stat: (B, C_static)

        x = self.stem(x_dyn)     # (B, 32, T, C_dyn)
        x = self.layer1(x)       # (B, 32, T, C_dyn)
        x = self.pool1(x)        # (B, 32, T/2, C/2)

        x = self.layer2(x)       # (B, 64, T/2, C/2)
        x = self.pool2(x)        # (B, 64, T/4, C/4)

        x = self.layer3(x)       # (B, 64, T/4, C/4)
        x = self.spatial_pool(x) # (B, 64, 4, 4)

        x = x.view(x.size(0), -1)   # (B, 64*4*4)

        x_stat = self.static_fc(x_stat)  # (B, 64)

        h = torch.cat([x, x_stat], dim=1)  # (B, cnn_out_dim + 64)
        logits = self.fc(h)                # (B, num_classes)
        return logits

# ==== SECTION 1: Colab + Kaggle setup ====
import os, json, getpass, pathlib, sys, time, shutil, subprocess, random
from pathlib import Path

import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


# Write kaggle.json securely via input (no manual upload of dataset needed)
KAGGLE_USERNAME = input("Enter your Kaggle username: ").strip()
KAGGLE_KEY = getpass.getpass("Enter your Kaggle API key (will be hidden): ").strip()

os.makedirs(Path.home()/".kaggle", exist_ok=True)
with open(Path.home()/".kaggle"/"kaggle.json", "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod(Path.home()/".kaggle"/"kaggle.json", 0o600)

print("Kaggle is configured.")

# --- (Optional) GPU check ---
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


COMPETITION = "an2dl2526c1"
assert COMPETITION, "Competition slug is required."

# --- Download competition files ---
!kaggle competitions files -c $COMPETITION
!kaggle competitions download -c $COMPETITION -p ./. -w
!mkdir -p ./data_extract && unzip -o "./*.zip" -d ./data_extract

# Sanity: list extracted files
print("\nExtracted files:")
!ls -l ./data_extract

# Expected file names (as described by the course page)
TRAIN_FILE = "pirate_pain_train.csv"
LABELS_FILE = "pirate_pain_train_labels.csv"
TEST_FILE  = "pirate_pain_test.csv"
SAMPLE_SUB = "sample_submission.csv"

for fname in [TRAIN_FILE, LABELS_FILE, TEST_FILE, SAMPLE_SUB]:
    path = Path("./data_extract")/fname
    if not path.exists():
        print(f"WARNING: {fname} not found in ./data_extract. Please check filenames on the competition page.")
# ==== SECTION 3: Data loading & basic checks ====
import pandas as pd
import numpy as np

train_df = pd.read_csv(Path("./data_extract") / TRAIN_FILE)
labels_df = pd.read_csv(Path("./data_extract") / LABELS_FILE)
test_df  = pd.read_csv(Path("./data_extract") / TEST_FILE)

# Sanity check: required columns
assert "sample_index" in train_df.columns and "time" in train_df.columns, "Missing sample_index/time in train_df"
assert "sample_index" in test_df.columns and "time" in test_df.columns, "Missing sample_index/time in test_df"
assert "label" in labels_df.columns and "sample_index" in labels_df.columns, "Missing label/sample_index in labels_df"

# Sort by sample_index, then time to ensure correct temporal ordering
train_df = train_df.sort_values(["sample_index", "time"]).reset_index(drop=True)
test_df  = test_df.sort_values(["sample_index", "time"]).reset_index(drop=True)

# Determine feature columns programmatically (exclude id/time)
exclude_cols = {"sample_index", "time"}
feature_cols = [c for c in train_df.columns if c not in exclude_cols]
print("Num features:", len(feature_cols), "->", feature_cols[:10], "...")

# Verify time length per sample
steps_per_sample = train_df.groupby("sample_index")["time"].nunique()
print("Time steps stats:", steps_per_sample.describe())
T_EXPECTED = 160
if steps_per_sample.nunique() != 1 or steps_per_sample.iloc[0] != T_EXPECTED:
    print("WARNING: Not all samples have the same number of time steps or not equal to 160.")
else:
    print("All samples have 160 time steps.")

print("Train shape:", train_df.shape, "Test shape:", test_df.shape)

# ==== SECTION 4: Static vs dynamic feature separation ====
static_features = ["n_legs", "n_hands", "n_eyes"]
dynamic_features = [c for c in feature_cols if c not in static_features]

print("Static features:", static_features)
print("Dynamic feature count:", len(dynamic_features))

# Inspect unique values in static columns (string-encoded pirates)
for col in static_features:
    print(col, "->", train_df[col].unique())

# Map text values to numeric counts
text2num = {
    "two": 2.0,
    "one+peg_leg": 1.0,
    "one+hook_hand": 1.0,
    "one+eye_patch": 1.0,
    # If there are other variants like "one", "zero", add here:
    "one": 1.0,
    "zero": 0.0,
}

def convert_static_columns(df, static_cols):
    for col in static_cols:
        df[col] = df[col].map(text2num).fillna(0.0).astype(np.float32)
    return df

train_df = convert_static_columns(train_df, static_features)
test_df  = convert_static_columns(test_df, static_features)

print("\nAfter static conversion:")
print(train_df[static_features].head())

# ==== SECTION 5: Convert DataFrame -> (sample_num, timestamp_num, channel_num) + static ====
def df_to_3d_with_static(df, dynamic_cols, static_cols):
    """
    df: has rows identified by (sample_index, time)
    returns:
        series_3d: (sample_num, timestamp_num, channel_num_dyn)
        static_2d: (sample_num, channel_num_static)
        sample_ids: (sample_num,)
    """
    groups = df.groupby("sample_index")

    sample_ids = []
    series_list = []   # each: (T, C_dyn)
    static_list = []   # each: (C_static,)

    for sid, g in groups:
        g = g.sort_values("time")

        # Dynamic time-series matrix (timestamp_num, channel_num_dyn)
        X_dyn = g[dynamic_cols].to_numpy(dtype=np.float32)

        # Static vector (same for all timesteps → take first row)
        X_static = g[static_cols].iloc[0].to_numpy(dtype=np.float32)

        sample_ids.append(sid)
        series_list.append(X_dyn)
        static_list.append(X_static)

    series_3d = np.stack(series_list, axis=0)   # (sample_num, timestamp_num, channel_num_dyn)
    static_2d = np.stack(static_list, axis=0)   # (sample_num, channel_num_static)

    return series_3d, static_2d, np.array(sample_ids)

X_train_dyn, X_train_static, train_ids = df_to_3d_with_static(
    train_df, dynamic_features, static_features
)
X_test_dyn, X_test_static, test_ids = df_to_3d_with_static(
    test_df, dynamic_features, static_features
)

print("Dynamic train shape:", X_train_dyn.shape)   # (N, T, C_dyn)
print("Static  train shape:", X_train_static.shape) # (N, C_static)

# ==== SECTION 6: Label mapping (fixed order: no_pain=0, low_pain=1, high_pain=2) ====
label_order = ["no_pain", "low_pain", "high_pain"]
label2id = {name: idx for idx, name in enumerate(label_order)}
id2label = {idx: name for idx, name in enumerate(label_order)}

print("Label → ID mapping:", label2id)

labels_map = labels_df.set_index("sample_index")["label"].to_dict()
y_train = np.array([label2id[labels_map[sid]] for sid in train_ids], dtype=np.int64)

print("y_train shape:", y_train.shape)
print("Unique y_train values:", np.unique(y_train))

# ==== SECTION 7: Correlation-based channel ordering for dynamic features ====
# Flatten over samples and time: (N, T, C_dyn) -> (N*T, C_dyn)
N, T, C_dyn = X_train_dyn.shape
X_flat = X_train_dyn.reshape(-1, C_dyn)  # (N*T, C_dyn)

# Compute correlation matrix between channels
corr = np.corrcoef(X_flat, rowvar=False)  # (C_dyn, C_dyn)
abs_corr = np.abs(corr)

# Greedy ordering: place highly correlated channels next to each other
remaining = set(range(C_dyn))
avg_corr = abs_corr.mean(axis=1)
start_idx = int(avg_corr.argmax())

order = [start_idx]
remaining.remove(start_idx)

while remaining:
    last = order[-1]
    next_idx = max(remaining, key=lambda j: abs_corr[last, j])
    order.append(next_idx)
    remaining.remove(next_idx)

order = np.array(order, dtype=int)
print("Original dynamic feature order:", dynamic_features[:10], "...")
dynamic_features = [dynamic_features[i] for i in order]
print("Reordered dynamic feature order (first 10):", dynamic_features[:10], "...")

# Apply this ordering to dynamic tensors
X_train_dyn = X_train_dyn[:, :, order]   # (N, T, C_dyn_reordered)
X_test_dyn  = X_test_dyn[:, :, order]

# ==== SECTION 8: Normalization (dynamic & static separately) ====
# Dynamic normalization over (N, T) for each dynamic channel
dyn_mean = X_train_dyn.mean(axis=(0, 1), keepdims=True)
dyn_std  = X_train_dyn.std(axis=(0, 1), keepdims=True) + 1e-6

X_train_dyn = (X_train_dyn - dyn_mean) / dyn_std
X_test_dyn  = (X_test_dyn  - dyn_mean) / dyn_std

# Static normalization over N for each static channel
stat_mean = X_train_static.mean(axis=0, keepdims=True)
stat_std  = X_train_static.std(axis=0, keepdims=True) + 1e-6

X_train_static = (X_train_static - stat_mean) / stat_std
X_test_static  = (X_test_static  - stat_mean) / stat_std

print("Static normalized sample:", X_train_static[:5])

# ==== SECTION 9: Stratified train/validation split ====
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tr_idx, va_idx = list(skf.split(train_ids, y_train))[0]

X_tr_dyn, X_va_dyn = X_train_dyn[tr_idx], X_train_dyn[va_idx]
X_tr_stat, X_va_stat = X_train_static[tr_idx], X_train_static[va_idx]
y_tr, y_va = y_train[tr_idx], y_train[va_idx]

print("Train dyn:", X_tr_dyn.shape, "Val dyn:", X_va_dyn.shape)
print("Train stat:", X_tr_stat.shape, "Val stat:", X_va_stat.shape)

# ==== SECTION 10: PyTorch Dataset & Dataloaders (for Conv2D) ====
from torch.utils.data import Dataset, DataLoader

class TimeSeries2DDataset(Dataset):
    def __init__(self, X_dyn, X_stat, y=None):
        self.X_dyn = X_dyn          # (N, T, C_dyn)
        self.X_stat = X_stat        # (N, C_static)
        self.y = y                  # (N,) or None

    def __len__(self):
        return self.X_dyn.shape[0]

    def __getitem__(self, idx):
        x_dyn = self.X_dyn[idx]  # (T, C_dyn)
        # For Conv2d: (B, C_in, H, W) → here (1, T, C_dyn)
        x_dyn = torch.from_numpy(x_dyn).unsqueeze(0).contiguous()  # (1, T, C_dyn)

        x_stat = torch.from_numpy(self.X_stat[idx])  # (C_static,)

        if self.y is None:
            return x_dyn, x_stat

        return x_dyn, x_stat, torch.tensor(self.y[idx], dtype=torch.long)

BATCH_SIZE = 64

train_ds = TimeSeries2DDataset(X_tr_dyn, X_tr_stat, y_tr)
val_ds   = TimeSeries2DDataset(X_va_dyn, X_va_stat, y_va)
test_ds  = TimeSeries2DDataset(X_test_dyn, X_test_static, None)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

# ==== SECTION 11: Model – Conv2D over (time × feature) + MLP over static ====
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score


static_dim = X_train_static.shape[-1]
num_classes = len(label_order)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = CNN2D_with_Static(static_dim, num_classes).to(device)

# ==== SECTION 12: Training utilities (train/eval loops) ====
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for x_dyn, x_stat, y in loader:
        x_dyn = x_dyn.to(device)
        x_stat = x_stat.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x_dyn, x_stat)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x_dyn.size(0)

    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_labels = []

    for x_dyn, x_stat, y in loader:
        x_dyn = x_dyn.to(device)
        x_stat = x_stat.to(device)
        y = y.to(device)

        logits = model(x_dyn, x_stat)
        loss = F.cross_entropy(logits, y, reduction='sum')
        probs = F.softmax(logits, dim=1)

        total_loss += loss.item()
        all_probs.append(probs.cpu().numpy())
        all_labels.append(y.cpu().numpy())

    all_probs = np.concatenate(all_probs, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    preds = all_probs.argmax(axis=1)
    macro_f1 = f1_score(all_labels, preds, average="macro")

    return total_loss / len(loader.dataset), macro_f1

# ==== SECTION 13: Train with early stopping on validation macro-F1 ====
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

best_val_f1 = -1.0
best_state = None
patience = 8
no_improve = 0
EPOCHS = 60

for epoch in range(1, EPOCHS + 1):
    tr_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    va_loss, va_f1 = evaluate(model, val_loader)

    print(f"Epoch {epoch:02d} | train_loss={tr_loss:.4f} | val_loss={va_loss:.4f} | val_macroF1={va_f1:.4f}")

    if va_f1 > best_val_f1 + 1e-5:
        best_val_f1 = va_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print("Early stopping.")
            break

if best_state is not None:
    model.load_state_dict(best_state)
    # save best model (before full-train)
    torch.save(model.state_dict(), "model_best_val.pt")
    print("Saved best val model to model_best_val.pt")

print("Best val macro-F1:", best_val_f1)

# ==== SECTION 14: (Optional) Fine-tune on full train ====
full_ds = TimeSeries2DDataset(X_train_dyn, X_train_static, y_train)
full_loader = DataLoader(full_ds, batch_size=BATCH_SIZE, shuffle=True)

optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
FINE_EPOCHS = 5

for epoch in range(1, FINE_EPOCHS + 1):
    tr_loss = train_one_epoch(model, full_loader, optimizer, criterion)
    print(f"[Full train] Epoch {epoch:02d} | loss={tr_loss:.4f}")


# save model AFTER full-train on all training data
torch.save(model.state_dict(), "model_full_train.pt")
print("Saved full-train model to model_full_train.pt")


Torch version: 2.8.0+cu126
CUDA available: True


In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# Pirate Pain – Preprocessing + CNN + BiGRU + MHA + Attn + Static
# With CLASS UNDERSAMPLING, training curves, F1 on train/val, confusion matrices,
# best-val model + full-train fine-tune + two submissions
# ============================================================

# ==== SECTION 0: Install dependencies (Colab) ====
!pip -q install kaggle scikit-learn

# ==== SECTION 1: Colab + Kaggle setup ====
import os, json, getpass, random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# Configure Kaggle API
KAGGLE_USERNAME = input("Enter your Kaggle username: ").strip()
KAGGLE_KEY = getpass.getpass("Enter your Kaggle API key (will be hidden): ").strip()

os.makedirs(Path.home() / ".kaggle", exist_ok=True)
with open(Path.home() / ".kaggle" / "kaggle.json", "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod(Path.home() / ".kaggle" / "kaggle.json", 0o600)

print("Kaggle is configured.")

# ==== SECTION 2: Download competition data ====
COMPETITION = "an2dl2526c1"   # Pirate Pain challenge
assert COMPETITION, "Competition slug is required."

!kaggle competitions files -c $COMPETITION
!kaggle competitions download -c $COMPETITION -p ./. -w
!mkdir -p ./data_extract && unzip -o "./*.zip" -d ./data_extract

print("\nExtracted files:")
!ls -l ./data_extract

TRAIN_FILE = "pirate_pain_train.csv"
LABELS_FILE = "pirate_pain_train_labels.csv"
TEST_FILE  = "pirate_pain_test.csv"
SAMPLE_SUB = "sample_submission.csv"

for fname in [TRAIN_FILE, LABELS_FILE, TEST_FILE, SAMPLE_SUB]:
    path = Path("./data_extract") / fname
    if not path.exists():
        print(f"WARNING: {fname} not found in ./data_extract. Please check filenames.")

# ==== SECTION 3: Data loading & basic checks ====
train_df = pd.read_csv(Path("./data_extract") / TRAIN_FILE)
labels_df = pd.read_csv(Path("./data_extract") / LABELS_FILE)
test_df  = pd.read_csv(Path("./data_extract") / TEST_FILE)

print("train_df shape:", train_df.shape)
print("labels_df shape:", labels_df.shape)
print("test_df shape:",  test_df.shape)

assert "sample_index" in train_df.columns and "time" in train_df.columns
assert "sample_index" in test_df.columns and "time" in test_df.columns
assert "label" in labels_df.columns and "sample_index" in labels_df.columns

train_df = train_df.sort_values(["sample_index", "time"]).reset_index(drop=True)
test_df  = test_df.sort_values(["sample_index", "time"]).reset_index(drop=True)

steps_per_sample = train_df.groupby("sample_index")["time"].nunique()
print("Time steps stats:", steps_per_sample.describe())
T_EXPECTED = 160
if steps_per_sample.nunique() != 1 or steps_per_sample.iloc[0] != T_EXPECTED:
    print("WARNING: Not all samples have the same number of time steps or not equal to 160.")
else:
    print("All samples have 160 time steps.")

# ==== SECTION 4: Static vs dynamic features (your style) ====
static_features = ["n_legs", "n_hands", "n_eyes"]

exclude_cols = {"sample_index", "time"}
feature_cols = [c for c in train_df.columns if c not in exclude_cols]

dynamic_features = [c for c in feature_cols if c not in static_features]

print("Static features:", static_features)
print("Dynamic feature count:", len(dynamic_features))
print("Sample dynamic features:", dynamic_features[:10])

for col in static_features:
    print(col, "->", train_df[col].unique())

text2num = {
    "two": 2.0,
    "one+peg_leg": 1.0,
    "one+hook_hand": 1.0,
    "one+eye_patch": 1.0,
    "one": 1.0,
    "zero": 0.0,
}

def convert_static_columns(df, static_cols):
    for col in static_cols:
        df[col] = df[col].map(text2num).fillna(0.0).astype(np.float32)
    return df

train_df = convert_static_columns(train_df, static_features)
test_df  = convert_static_columns(test_df, static_features)

print("\nStatic columns after conversion:")
print(train_df[static_features].head())

# ==== SECTION 5: DataFrame -> (N, T, C_dyn) + static (N, C_static) ====
def df_to_3d_with_static(df, dynamic_cols, static_cols):
    """
    df: rows indexed by (sample_index, time)
    Returns:
        X_dyn: (N, T, C_dyn)
        X_static: (N, C_static)
        sample_ids: (N,)
    """
    groups = df.groupby("sample_index")

    sample_ids = []
    series_list = []   # (T, C_dyn)
    static_list = []   # (C_static,)

    for sid, g in groups:
        g = g.sort_values("time")

        X_dyn = g[dynamic_cols].to_numpy(dtype=np.float32)            # (T, C_dyn)
        X_static = g[static_cols].iloc[0].to_numpy(dtype=np.float32)  # (C_static,)

        sample_ids.append(sid)
        series_list.append(X_dyn)
        static_list.append(X_static)

    X_dyn = np.stack(series_list, axis=0)      # (N, T, C_dyn)
    X_static = np.stack(static_list, axis=0)   # (N, C_static)
    sample_ids = np.array(sample_ids)

    return X_dyn, X_static, sample_ids

X_train_dyn, X_train_static, train_ids = df_to_3d_with_static(
    train_df, dynamic_features, static_features
)
X_test_dyn, X_test_static, test_ids = df_to_3d_with_static(
    test_df, dynamic_features, static_features
)

print("X_train_dyn:", X_train_dyn.shape)      # (N, T, C_dyn)
print("X_train_static:", X_train_static.shape)
print("X_test_dyn:", X_test_dyn.shape)
print("X_test_static:", X_test_static.shape)

# ==== SECTION 6: Label mapping (no_pain=0, low_pain=1, high_pain=2) ====
label_order = ["no_pain", "low_pain", "high_pain"]
label2id = {name: idx for idx, name in enumerate(label_order)}
id2label = {idx: name for idx, name in enumerate(label_order)}

print("Label → ID mapping:", label2id)

labels_map = labels_df.set_index("sample_index")["label"].to_dict()
y_train = np.array([label2id[labels_map[sid]] for sid in train_ids], dtype=np.int64)

print("y_train shape:", y_train.shape)
print("Unique labels:", np.unique(y_train))

# ==== SECTION 7: Class distribution (overall) ====
unique, counts = np.unique(y_train, return_counts=True)
print("\nClass distribution (overall train):")
for cls_id, cnt in zip(unique, counts):
    cls_name = id2label[cls_id]
    frac = cnt / len(y_train)
    print(f"  class {cls_id} ({cls_name}): {cnt} samples ({frac:.2%})")

# ==== SECTION 8: Normalization (dynamic & static separately) ====
dyn_mean = X_train_dyn.mean(axis=(0, 1), keepdims=True)
dyn_std  = X_train_dyn.std(axis=(0, 1), keepdims=True) + 1e-6
X_train_dyn = (X_train_dyn - dyn_mean) / dyn_std
X_test_dyn  = (X_test_dyn  - dyn_mean) / dyn_std

stat_mean = X_train_static.mean(axis=0, keepdims=True)
stat_std  = X_train_static.std(axis=0, keepdims=True) + 1e-6
X_train_static = (X_train_static - stat_mean) / stat_std
X_test_static  = (X_test_static  - stat_mean) / stat_std

print("Static normalized sample:", X_train_static[:5])

# ==== SECTION 9: Stratified train/val split ====
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tr_idx, va_idx = list(skf.split(train_ids, y_train))[0]

X_tr_dyn, X_va_dyn = X_train_dyn[tr_idx], X_train_dyn[va_idx]
X_tr_stat, X_va_stat = X_train_static[tr_idx], X_train_static[va_idx]
y_tr, y_va = y_train[tr_idx], y_train[va_idx]

print("\nClass distribution (train split BEFORE undersampling):")
u_tr, c_tr = np.unique(y_tr, return_counts=True)
for cls_id, cnt in zip(u_tr, c_tr):
    print(f"  class {cls_id} ({id2label[cls_id]}): {cnt} samples ({cnt/len(y_tr):.2%})")

print("\nClass distribution (val split):")
u_va, c_va = np.unique(y_va, return_counts=True)
for cls_id, cnt in zip(u_va, c_va):
    print(f"  class {cls_id} ({id2label[cls_id]}): {cnt} samples ({cnt/len(y_va):.2%})")

# ==== SECTION 10: Random undersampling on TRAIN split ====
rng = np.random.default_rng(42)
classes = np.unique(y_tr)
class_counts = np.bincount(y_tr)
min_count = class_counts[class_counts > 0].min()   # smallest non-zero count

balanced_indices = []
for c in classes:
    idx_c = np.where(y_tr == c)[0]
    chosen = rng.choice(idx_c, size=min_count, replace=False)
    balanced_indices.append(chosen)

balanced_indices = np.concatenate(balanced_indices)
rng.shuffle(balanced_indices)

X_tr_dyn = X_tr_dyn[balanced_indices]
X_tr_stat = X_tr_stat[balanced_indices]
y_tr = y_tr[balanced_indices]

print("\nClass distribution (train split AFTER undersampling):")
u_tr2, c_tr2 = np.unique(y_tr, return_counts=True)
for cls_id, cnt in zip(u_tr2, c_tr2):
    print(f"  class {cls_id} ({id2label[cls_id]}): {cnt} samples ({cnt/len(y_tr):.2%})")

# ==== SECTION 11: Dataset & Dataloaders (NO augmentation) ====
from torch.utils.data import Dataset, DataLoader

class TimeSeriesSeqDataset(Dataset):
    def __init__(self, X_dyn, X_stat, y=None, augment=False):
        self.X_dyn = X_dyn          # (N, T, C_dyn)
        self.X_stat = X_stat        # (N, C_static)
        self.y = y                  # (N,) or None
        self.augment = augment      # will be False here

    def __len__(self):
        return self.X_dyn.shape[0]

    def _augment(self, x):
        # not used (augment=False)
        return x

    def __getitem__(self, idx):
        x_dyn = self.X_dyn[idx]          # (T, C_dyn)
        x_stat = self.X_stat[idx]        # (C_static,)

        if self.augment:
            x_dyn = self._augment(x_dyn)

        x_dyn = torch.from_numpy(x_dyn).float()
        x_stat = torch.from_numpy(x_stat).float()

        if self.y is None:
            return x_dyn, x_stat

        label = torch.tensor(self.y[idx], dtype=torch.long)
        return x_dyn, x_stat, label

BATCH_SIZE = 256

train_ds = TimeSeriesSeqDataset(X_tr_dyn, X_tr_stat, y_tr, augment=False)
val_ds   = TimeSeriesSeqDataset(X_va_dyn, X_va_stat, y_va, augment=False)
test_ds  = TimeSeriesSeqDataset(X_test_dyn, X_test_static, None, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=128, shuffle=False)

# ==== SECTION 12: Model – CNN + BiGRU + Multi-Head Attention + Additive Attention + Static ====
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

class AttentionPooling(nn.Module):
    """
    Additive attention over time.
    Input:  x -> (B, T, D)
    Output: context -> (B, D)
    """
    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim, dim)
        self.score = nn.Linear(dim, 1, bias=False)

    def forward(self, x, mask=None):
        # x: (B, T, D)
        h = torch.tanh(self.proj(x))          # (B, T, D)
        scores = self.score(h).squeeze(-1)    # (B, T)

        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))

        weights = torch.softmax(scores, dim=1)          # (B, T)
        context = torch.bmm(weights.unsqueeze(1), x)    # (B, 1, D)
        context = context.squeeze(1)                    # (B, D)
        return context

class CNNGRUWithAttention(nn.Module):
    """
    Dynamic branch:
      - Conv1d stem over time
      - 2-layer BiGRU
      - Multihead self-attention + residual + LayerNorm
      - Triple pooling: mean, max, additive attention
    Static branch:
      - Small MLP on static features
    Then concat and classify.
    """
    def __init__(self, dyn_channels, static_dim, num_classes):
        super().__init__()

        # 1) Conv1d stem: input (B, C_dyn, T)
        self.conv = nn.Sequential(
            nn.Conv1d(
                in_channels=dyn_channels,
                out_channels=64,
                kernel_size=7,
                padding=3,
            ),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.MaxPool1d(kernel_size=2),   # T -> T/2
            nn.Dropout(0.3),
        )

        # 2) BiGRU
        self.gru = nn.GRU(
            input_size=128,
            hidden_size=160,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3,
        )  # output dim = 320
        self.gru_ln = nn.LayerNorm(320)

        # 3) Multi-head self-attention + residual
        self.mha = nn.MultiheadAttention(
            embed_dim=320,
            num_heads=4,
            dropout=0.25,
            batch_first=True,
        )
        self.mha_ln = nn.LayerNorm(320)

        # 4) Additive attention pooling
        self.attn_pool = AttentionPooling(dim=320)

        # 5) Static branch
        if static_dim > 0:
            self.static_mlp = nn.Sequential(
                nn.Linear(static_dim, 8),
                nn.ReLU(),
            )
            static_out_dim = 8
        else:
            self.static_mlp = None
            static_out_dim = 0

        # 6) Classifier
        seq_repr_dim = 320 * 3   # mean + max + attn
        total_dim = seq_repr_dim + static_out_dim

        self.dropout_head = nn.Dropout(0.4)

        self.fc = nn.Sequential(
            nn.Linear(total_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x_dyn, x_stat):
        """
        x_dyn:  (B, T, C_dyn)
        x_stat: (B, C_static)
        """
        # (B, T, C_dyn) -> (B, C_dyn, T)
        x = x_dyn.permute(0, 2, 1)            # (B, C_dyn, T)

        # CNN stem
        x = self.conv(x)                      # (B, 128, T')
        x = x.permute(0, 2, 1)                # (B, T', 128)

        # BiGRU
        out, _ = self.gru(x)                  # (B, T', 320)
        out = self.gru_ln(out)                # (B, T', 320)

        # Multi-head self-attention + residual
        attn_out, _ = self.mha(out, out, out) # (B, T', 320)
        out = self.mha_ln(out + attn_out)     # (B, T', 320)

        # Triple pooling
        mean_pool = out.mean(dim=1)           # (B, 320)
        max_pool, _ = out.max(dim=1)          # (B, 320)
        attn_context = self.attn_pool(out)    # (B, 320)

        x_seq = torch.cat([mean_pool, max_pool, attn_context], dim=1)  # (B, 960)
        x_seq = self.dropout_head(x_seq)

        # Static branch
        if self.static_mlp is not None:
            s = self.static_mlp(x_stat)       # (B, 8)
            z = torch.cat([x_seq, s], dim=1)  # (B, 968)
        else:
            z = x_seq                         # (B, 960)

        logits = self.fc(z)                   # (B, num_classes)
        return logits

dyn_channels = X_train_dyn.shape[-1]
static_dim = X_train_static.shape[-1]
num_classes = len(label_order)

model = CNNGRUWithAttention(
    dyn_channels=dyn_channels,
    static_dim=static_dim,
    num_classes=num_classes,
).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=4e-5)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3
)

# ==== SECTION 13: Train with early stopping + history & plots + confusion matrices ====
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for x_dyn, x_stat, y in loader:
        x_dyn = x_dyn.to(device)
        x_stat = x_stat.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x_dyn, x_stat)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * x_dyn.size(0)

    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_targets = []

    for x_dyn, x_stat, y in loader:
        x_dyn = x_dyn.to(device)
        x_stat = x_stat.to(device)
        y = y.to(device)

        logits = model(x_dyn, x_stat)
        loss = criterion(logits, y)
        total_loss += loss.item() * x_dyn.size(0)

        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.append(preds)
        all_targets.append(y.cpu().numpy())

    total_loss /= len(loader.dataset)
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    macro_f1 = f1_score(all_targets, all_preds, average="macro")
    return total_loss, macro_f1, all_targets, all_preds

n_epochs = 120
best_val_f1 = -1.0
best_state = None
patience = 12
no_improve = 0

history_train_loss = []
history_val_loss   = []
history_train_f1   = []
history_val_f1     = []

for epoch in range(1, n_epochs + 1):
    # Train one epoch (optimization loss)
    train_loss_opt = train_one_epoch(model, train_loader, optimizer, criterion)

    # Evaluate on train and val (to get F1 for both)
    train_loss_eval, train_f1, train_targets, train_preds = evaluate(model, train_loader, criterion)
    val_loss, val_f1, val_targets, val_preds = evaluate(model, val_loader, criterion)

    scheduler.step(val_f1)

    history_train_loss.append(train_loss_opt)
    history_val_loss.append(val_loss)
    history_train_f1.append(train_f1)
    history_val_f1.append(val_f1)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss_opt={train_loss_opt:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"train_macroF1={train_f1:.4f} | "
        f"val_macroF1={val_f1:.4f}"
    )

    # Early stopping based on val F1
    if val_f1 > best_val_f1 + 1e-5:
        best_val_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print("Early stopping.")
            break

print("Best val macro-F1:", best_val_f1)

# Load and save best-val model
if best_state is not None:
    model.load_state_dict(best_state)
    torch.save(model.state_dict(), "model_best_val.pt")
    print("Saved best-val model to model_best_val.pt")

# ---- 13-B: Plot training curves (loss + F1) ----
epochs_ran = range(1, len(history_train_loss) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_ran, history_train_loss, label="Train Loss (opt step)")
plt.plot(epochs_ran, history_val_loss,   label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Val Loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs_ran, history_train_f1, label="Train Macro-F1")
plt.plot(epochs_ran, history_val_f1,   label="Val Macro-F1")
plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.title("Train vs Val Macro-F1")
plt.legend()
plt.grid(True)
plt.show()

# ---- 13-C: Confusion matrices for TRAIN and VAL using best model ----
@torch.no_grad()
def get_preds_and_targets(model, loader):
    model.eval()
    all_preds = []
    all_targets = []
    for x_dyn, x_stat, y in loader:
        x_dyn = x_dyn.to(device)
        x_stat = x_stat.to(device)
        y = y.to(device)

        logits = model(x_dyn, x_stat)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.append(preds)
        all_targets.append(y.cpu().numpy())
    return np.concatenate(all_targets), np.concatenate(all_preds)

best_model = CNNGRUWithAttention(
    dyn_channels=dyn_channels,
    static_dim=static_dim,
    num_classes=num_classes,
).to(device)
best_model.load_state_dict(torch.load("model_best_val.pt", map_location=device))

# Train confusion matrix
train_targets_cm, train_preds_cm = get_preds_and_targets(best_model, train_loader)
cm_train = confusion_matrix(train_targets_cm, train_preds_cm, labels=[0, 1, 2])

plt.figure(figsize=(5, 4))
disp_train = ConfusionMatrixDisplay(
    confusion_matrix=cm_train,
    display_labels=[id2label[i] for i in [0, 1, 2]]
)
disp_train.plot(values_format="d", cmap="Blues", colorbar=False)
plt.title("Confusion Matrix - TRAIN (best model)")
plt.show()

# Val confusion matrix
val_targets_cm, val_preds_cm = get_preds_and_targets(best_model, val_loader)
cm_val = confusion_matrix(val_targets_cm, val_preds_cm, labels=[0, 1, 2])

plt.figure(figsize=(5, 4))
disp_val = ConfusionMatrixDisplay(
    confusion_matrix=cm_val,
    display_labels=[id2label[i] for i in [0, 1, 2]]
)
disp_val.plot(values_format="d", cmap="Blues", colorbar=False)
plt.title("Confusion Matrix - VAL (best model)")
plt.show()

# ==== SECTION 14: Full-train (with undersampling on full train) & save full model ====
rng_full = np.random.default_rng(123)
classes_full = np.unique(y_train)
counts_full = np.bincount(y_train)
min_count_full = counts_full[counts_full > 0].min()

balanced_full_indices = []
for c in classes_full:
    idx_c = np.where(y_train == c)[0]
    chosen = rng_full.choice(idx_c, size=min_count_full, replace=False)
    balanced_full_indices.append(chosen)

balanced_full_indices = np.concatenate(balanced_full_indices)
rng_full.shuffle(balanced_full_indices)

X_full_dyn = X_train_dyn[balanced_full_indices]
X_full_stat = X_train_static[balanced_full_indices]
y_full = y_train[balanced_full_indices]

print("\nClass distribution (FULL train AFTER undersampling):")
u_f, c_f = np.unique(y_full, return_counts=True)
for cls_id, cnt in zip(u_f, c_f):
    print(f"  class {cls_id} ({id2label[cls_id]}): {cnt} samples ({cnt/len(y_full):.2%})")

full_ds = TimeSeriesSeqDataset(X_full_dyn, X_full_stat, y_full, augment=False)
full_loader = DataLoader(full_ds, batch_size=BATCH_SIZE, shuffle=True)

model_full = CNNGRUWithAttention(
    dyn_channels=dyn_channels,
    static_dim=static_dim,
    num_classes=num_classes,
).to(device)

state_best = torch.load("model_best_val.pt", map_location=device)
model_full.load_state_dict(state_best)

optimizer_full = torch.optim.AdamW(model_full.parameters(), lr=3e-4, weight_decay=4e-5)

FINE_EPOCHS = 5
for epoch in range(1, FINE_EPOCHS + 1):
    loss_full = train_one_epoch(model_full, full_loader, optimizer_full, criterion)
    print(f"[Full train] Epoch {epoch:02d} | loss={loss_full:.4f}")

torch.save(model_full.state_dict(), "model_full_train.pt")
print("Saved full-train model to model_full_train.pt")

# ==== SECTION 15: Inference & submissions for both models ====
@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    all_probs = []
    for x_dyn, x_stat in loader:
        x_dyn = x_dyn.to(device)
        x_stat = x_stat.to(device)

        logits = model(x_dyn, x_stat)
        probs = F.softmax(logits, dim=1)
        all_probs.append(probs.cpu().numpy())
    return np.concatenate(all_probs, axis=0)

# 1) Best-val model submission
print("\n=== Inference with best-val model ===")
model_best = CNNGRUWithAttention(
    dyn_channels=dyn_channels,
    static_dim=static_dim,
    num_classes=num_classes,
).to(device)
model_best.load_state_dict(torch.load("model_best_val.pt", map_location=device))

test_probs_best = predict_proba(model_best, test_loader)
test_pred_ids_best = test_probs_best.argmax(axis=1)
test_pred_labels_best = [id2label[i] for i in test_pred_ids_best]

submission_best = pd.DataFrame({
    "sample_index": test_ids,
    "label": test_pred_labels_best
}).sort_values("sample_index")

submission_best.to_csv("submission_best_val.csv", index=False)
print(submission_best.head())
print("Saved submission_best_val.csv")

# 2) Full-train model submission
print("\n=== Inference with full-train model ===")
model_full_load = CNNGRUWithAttention(
    dyn_channels=dyn_channels,
    static_dim=static_dim,
    num_classes=num_classes,
).to(device)
model_full_load.load_state_dict(torch.load("model_full_train.pt", map_location=device))

test_probs_full = predict_proba(model_full_load, test_loader)
test_pred_ids_full = test_probs_full.argmax(axis=1)
test_pred_labels_full = [id2label[i] for i in test_pred_ids_full]

submission_full = pd.DataFrame({
    "sample_index": test_ids,
    "label": test_pred_labels_full
}).sort_values("sample_index")

submission_full.to_csv("submission_full_train.csv", index=False)
print(submission_full.head())
print("Saved submission_full_train.csv")


In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# Pirate Pain – CNN + RNN (GRU/LSTM) Hybrid + Static Features
# ============================================================

# ==== SECTION 0: Install dependencies (Colab) ====
!pip -q install kaggle scikit-learn

# Global RNN type: choose "gru" or "lstm"
rnn_type = "lstm"   # or "lstm"

# ==== SECTION 1: Colab + Kaggle setup ====
import os, json, getpass, pathlib, sys, time, shutil, subprocess, random
from pathlib import Path

import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# Configure Kaggle API
KAGGLE_USERNAME = input("Enter your Kaggle username: ").strip()
KAGGLE_KEY = getpass.getpass("Enter your Kaggle API key (will be hidden): ").strip()

os.makedirs(Path.home() / ".kaggle", exist_ok=True)
with open(Path.home() / ".kaggle" / "kaggle.json", "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod(Path.home() / ".kaggle" / "kaggle.json", 0o600)

print("Kaggle is configured.")

# ==== SECTION 2: Download competition data ====
COMPETITION = "an2dl2526c1"   # Pirate Pain challenge
assert COMPETITION, "Competition slug is required."

# List files on Kaggle
!kaggle competitions files -c $COMPETITION

# Download to current directory and unzip
!kaggle competitions download -c $COMPETITION -p ./. -w
!mkdir -p ./data_extract && unzip -o "./*.zip" -d ./data_extract

print("\nExtracted files:")
!ls -l ./data_extract

# Expected file names
TRAIN_FILE = "pirate_pain_train.csv"
LABELS_FILE = "pirate_pain_train_labels.csv"
TEST_FILE  = "pirate_pain_test.csv"
SAMPLE_SUB = "sample_submission.csv"

for fname in [TRAIN_FILE, LABELS_FILE, TEST_FILE, SAMPLE_SUB]:
    path = Path("./data_extract") / fname
    if not path.exists():
        print(f"WARNING: {fname} not found in ./data_extract. Please check filenames on the competition page.")

# ==== SECTION 3: Data loading & basic checks ====
import pandas as pd
import numpy as np

train_df = pd.read_csv(Path("./data_extract") / TRAIN_FILE)
labels_df = pd.read_csv(Path("./data_extract") / LABELS_FILE)
test_df  = pd.read_csv(Path("./data_extract") / TEST_FILE)

# Sanity check: required columns
assert "sample_index" in train_df.columns and "time" in train_df.columns, "Missing sample_index/time in train_df"
assert "sample_index" in test_df.columns and "time" in test_df.columns, "Missing sample_index/time in test_df"
assert "label" in labels_df.columns and "sample_index" in labels_df.columns, "Missing label/sample_index in labels_df"

# Sort by sample_index, then time to ensure correct temporal ordering
train_df = train_df.sort_values(["sample_index", "time"]).reset_index(drop=True)
test_df  = test_df.sort_values(["sample_index", "time"]).reset_index(drop=True)

# Determine feature columns programmatically (exclude id/time)
exclude_cols = {"sample_index", "time"}
feature_cols = [c for c in train_df.columns if c not in exclude_cols]
print("Num features:", len(feature_cols), "->", feature_cols[:10], "...")

# Verify time length per sample
steps_per_sample = train_df.groupby("sample_index")["time"].nunique()
print("Time steps stats:", steps_per_sample.describe())
T_EXPECTED = 160
if steps_per_sample.nunique() != 1 or steps_per_sample.iloc[0] != T_EXPECTED:
    print("WARNING: Not all samples have the same number of time steps or not equal to 160.")
else:
    print("All samples have 160 time steps.")

print("Train shape:", train_df.shape, "Test shape:", test_df.shape)

# ==== SECTION 4: Static vs dynamic feature separation ====
static_features = ["n_legs", "n_hands", "n_eyes"]
dynamic_features = [c for c in feature_cols if c not in static_features]

print("Static features:", static_features)
print("Dynamic feature count:", len(dynamic_features))

# Inspect unique values in static columns (string-encoded pirates)
for col in static_features:
    print(col, "->", train_df[col].unique())

# Map text values to numeric counts
text2num = {
    "two": 2.0,
    "one+peg_leg": 1.0,
    "one+hook_hand": 1.0,
    "one+eye_patch": 1.0,
    "one": 1.0,
    "zero": 0.0,
}

def convert_static_columns(df, static_cols):
    for col in static_cols:
        df[col] = df[col].map(text2num).fillna(0.0).astype(np.float32)
    return df

train_df = convert_static_columns(train_df, static_features)
test_df  = convert_static_columns(test_df, static_features)

print("\nAfter static conversion:")
print(train_df[static_features].head())

# ==== SECTION 5: Convert DataFrame -> (sample_num, timestamp_num, channel_num) + static ====
def df_to_3d_with_static(df, dynamic_cols, static_cols):
    """
    df: has rows identified by (sample_index, time)
    returns:
        series_3d: (N, T, C_dyn)
        static_2d: (N, C_static)
        sample_ids: (N,)
    """
    groups = df.groupby("sample_index")

    sample_ids = []
    series_list = []   # each: (T, C_dyn)
    static_list = []   # each: (C_static,)

    for sid, g in groups:
        g = g.sort_values("time")

        # Dynamic time-series matrix (T, C_dyn)
        X_dyn = g[dynamic_cols].to_numpy(dtype=np.float32)

        # Static vector (same for all timesteps → take first row)
        X_static = g[static_cols].iloc[0].to_numpy(dtype=np.float32)

        sample_ids.append(sid)
        series_list.append(X_dyn)
        static_list.append(X_static)

    series_3d = np.stack(series_list, axis=0)   # (N, T, C_dyn)
    static_2d = np.stack(static_list, axis=0)   # (N, C_static)

    return series_3d, static_2d, np.array(sample_ids)

X_train_dyn, X_train_static, train_ids = df_to_3d_with_static(
    train_df, dynamic_features, static_features
)
X_test_dyn, X_test_static, test_ids = df_to_3d_with_static(
    test_df, dynamic_features, static_features
)

print("Dynamic train shape:", X_train_dyn.shape)   # (N, T, C_dyn)
print("Static  train shape:", X_train_static.shape) # (N, C_static)

# ==== SECTION 6: Label mapping (fixed order: no_pain=0, low_pain=1, high_pain=2) ====
label_order = ["no_pain", "low_pain", "high_pain"]
label2id = {name: idx for idx, name in enumerate(label_order)}
id2label = {idx: name for idx, name in enumerate(label_order)}

print("Label → ID mapping:", label2id)

labels_map = labels_df.set_index("sample_index")["label"].to_dict()
y_train = np.array([label2id[labels_map[sid]] for sid in train_ids], dtype=np.int64)

print("y_train shape:", y_train.shape)
print("Unique y_train values:", np.unique(y_train))

# ==== SECTION 7: Normalization (dynamic & static separately) ====
# Dynamic normalization over (N, T) for each dynamic channel
dyn_mean = X_train_dyn.mean(axis=(0, 1), keepdims=True)
dyn_std  = X_train_dyn.std(axis=(0, 1), keepdims=True) + 1e-6

X_train_dyn = (X_train_dyn - dyn_mean) / dyn_std
X_test_dyn  = (X_test_dyn  - dyn_mean) / dyn_std

# Static normalization over N for each static channel
stat_mean = X_train_static.mean(axis=0, keepdims=True)
stat_std  = X_train_static.std(axis=0, keepdims=True) + 1e-6

X_train_static = (X_train_static - stat_mean) / stat_std
X_test_static  = (X_test_static  - stat_mean) / stat_std

print("Static normalized sample:", X_train_static[:5])

# ==== SECTION 8: Stratified train/validation split ====
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tr_idx, va_idx = list(skf.split(train_ids, y_train))[0]

X_tr_dyn, X_va_dyn = X_train_dyn[tr_idx], X_train_dyn[va_idx]
X_tr_stat, X_va_stat = X_train_static[tr_idx], X_train_static[va_idx]
y_tr, y_va = y_train[tr_idx], y_train[va_idx]

print("Train dyn:", X_tr_dyn.shape, "Val dyn:", X_va_dyn.shape)
print("Train stat:", X_tr_stat.shape, "Val stat:", X_va_stat.shape)

# ==== SECTION 9: PyTorch Dataset & Dataloaders (for CNN+RNN) ====
from torch.utils.data import Dataset, DataLoader

class TimeSeriesSeqDataset(Dataset):
    def __init__(self, X_dyn, X_stat, y=None):
        self.X_dyn = X_dyn          # (N, T, C_dyn)
        self.X_stat = X_stat        # (N, C_static)
        self.y = y                  # (N,) or None

    def __len__(self):
        return self.X_dyn.shape[0]

    def __getitem__(self, idx):
        x_dyn = self.X_dyn[idx]  # (T, C_dyn)
        x_dyn = torch.from_numpy(x_dyn).contiguous().float()  # (T, C_dyn)

        x_stat = torch.from_numpy(self.X_stat[idx]).float()   # (C_static,)

        if self.y is None:
            return x_dyn, x_stat

        return x_dyn, x_stat, torch.tensor(self.y[idx], dtype=torch.long)

BATCH_SIZE = 64

train_ds = TimeSeriesSeqDataset(X_tr_dyn, X_tr_stat, y_tr)
val_ds   = TimeSeriesSeqDataset(X_va_dyn, X_va_stat, y_va)
test_ds  = TimeSeriesSeqDataset(X_test_dyn, X_test_static, None)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

# ==== SECTION 10: Model – CNN (1D) over time + RNN (GRU/LSTM) + Static MLP ====
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score

class CNNRNN_with_Static(nn.Module):
    """
    Hybrid model:
      - 1D CNN over time on dynamic features
      - RNN (GRU or LSTM) over the sequence of CNN features
      - MLP on static features (n_legs, n_hands, n_eyes, ...)
      - Concatenate [RNN output, static embedding] → classifier
    Uses global rnn_type ("gru" or "lstm").
    """
    def __init__(
        self,
        dyn_channels: int,        # number of dynamic features (C_dyn)
        static_dim: int,          # number of static features (C_static)
        num_classes: int,
        cnn_hidden: int = 128,    # output channels of the CNN block
        rnn_hidden: int = 128,    # hidden size of RNN
        rnn_layers: int = 1,      # number of RNN layers
        bidirectional: bool = True,
    ):
        super().__init__()

        # 1D CNN over time: input (B, C_dyn, T)
        self.cnn = nn.Sequential(
            nn.Conv1d(dyn_channels, 64, kernel_size=7, padding=3, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),

            nn.Conv1d(64, 64, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),

            nn.Conv1d(64, cnn_hidden, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm1d(cnn_hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
        )

        self.rnn_hidden = rnn_hidden
        self.bidirectional = bidirectional

        # Use global rnn_type
        global rnn_type
        self.rnn_type = rnn_type.lower().strip()
        assert self.rnn_type in ["gru", "lstm"], "rnn_type must be 'gru' or 'lstm'"

        rnn_input_dim = cnn_hidden
        num_directions = 2 if bidirectional else 1

        # RNN over temporal sequence of CNN features: input (B, T, cnn_hidden)
        if self.rnn_type == "gru":
            self.rnn = nn.GRU(
                input_size=rnn_input_dim,
                hidden_size=rnn_hidden,
                num_layers=rnn_layers,
                batch_first=True,
                bidirectional=bidirectional,
            )
        else:  # LSTM
            self.rnn = nn.LSTM(
                input_size=rnn_input_dim,
                hidden_size=rnn_hidden,
                num_layers=rnn_layers,
                batch_first=True,
                bidirectional=bidirectional,
            )

        # Static features MLP
        self.static_fc = nn.Sequential(
            nn.Linear(static_dim, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
        )

        # Final classifier
        rnn_out_dim = rnn_hidden * num_directions
        self.fc = nn.Sequential(
            nn.Linear(rnn_out_dim + 64, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x_dyn, x_stat):
        """
        x_dyn:  (B, T, C_dyn)
        x_stat: (B, C_static)
        """
        # CNN expects (B, C_dyn, T)
        x = x_dyn.permute(0, 2, 1)           # (B, C_dyn, T)
        x = self.cnn(x)                      # (B, cnn_hidden, T)

        # RNN expects (B, T, D)
        x = x.permute(0, 2, 1)              # (B, T, cnn_hidden)

        if self.rnn_type == "gru":
            rnn_out, _ = self.rnn(x)        # (B, T, rnn_hidden * dirs)
        else:  # LSTM
            rnn_out, _ = self.rnn(x)        # (B, T, rnn_hidden * dirs)

        # Use last time step; (you can experiment with mean pooling over T too)
        last = rnn_out[:, -1, :]            # (B, rnn_hidden * dirs)

        # Static branch
        stat = self.static_fc(x_stat)       # (B, 64)

        # Concatenate and classify
        h = torch.cat([last, stat], dim=1)  # (B, rnn_hidden*dirs + 64)
        logits = self.fc(h)                 # (B, num_classes)
        return logits

dyn_channels = X_train_dyn.shape[-1]
static_dim = X_train_static.shape[-1]
num_classes = len(label_order)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("Using RNN type:", rnn_type)

model = CNNRNN_with_Static(
    dyn_channels=dyn_channels,
    static_dim=static_dim,
    num_classes=num_classes,
    cnn_hidden=128,
    rnn_hidden=128,
    rnn_layers=1,
    bidirectional=True,
).to(device)

# ==== SECTION 11: Training utilities (train/eval loops) ====
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for x_dyn, x_stat, y in loader:
        x_dyn = x_dyn.to(device)       # (B, T, C_dyn)
        x_stat = x_stat.to(device)     # (B, C_static)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x_dyn, x_stat)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x_dyn.size(0)

    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_labels = []

    for x_dyn, x_stat, y in loader:
        x_dyn = x_dyn.to(device)
        x_stat = x_stat.to(device)
        y = y.to(device)

        logits = model(x_dyn, x_stat)
        loss = F.cross_entropy(logits, y, reduction='sum')
        probs = F.softmax(logits, dim=1)

        total_loss += loss.item()
        all_probs.append(probs.cpu().numpy())
        all_labels.append(y.cpu().numpy())

    all_probs = np.concatenate(all_probs, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    preds = all_probs.argmax(axis=1)
    macro_f1 = f1_score(all_labels, preds, average="macro")

    return total_loss / len(loader.dataset), macro_f1

# ==== SECTION 12: Train with early stopping on validation macro-F1 ====
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

best_val_f1 = -1.0
best_state = None
patience = 8
no_improve = 0
EPOCHS = 60

for epoch in range(1, EPOCHS + 1):
    tr_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    va_loss, va_f1 = evaluate(model, val_loader)

    print(f"Epoch {epoch:02d} | train_loss={tr_loss:.4f} | val_loss={va_loss:.4f} | val_macroF1={va_f1:.4f}")

    if va_f1 > best_val_f1 + 1e-5:
        best_val_f1 = va_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print("Early stopping.")
            break

# Load best and save it
if best_state is not None:
    model.load_state_dict(best_state)
    torch.save(model.state_dict(), "model_best_val.pt")
    print("Saved best val model to model_best_val.pt")

print("Best val macro-F1:", best_val_f1)

# ==== SECTION 13: (Optional) Fine-tune on full train ====
full_ds = TimeSeriesSeqDataset(X_train_dyn, X_train_static, y_train)
full_loader = DataLoader(full_ds, batch_size=BATCH_SIZE, shuffle=True)

optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
FINE_EPOCHS = 5

for epoch in range(1, FINE_EPOCHS + 1):
    tr_loss = train_one_epoch(model, full_loader, optimizer, criterion)
    print(f"[Full train] Epoch {epoch:02d} | loss={tr_loss:.4f}")

# Save full-train model
torch.save(model.state_dict(), "model_full_train.pt")
print("Saved full-train model to model_full_train.pt")


In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# Pirate Pain – preprocessing + CNN + BiGRU + MHA + Attn + Static
# With:
#  - CLASS OVERSAMPLING (WeightedRandomSampler)
#  - LIGHT AUGMENTATION on train
#  - 5-fold Stratified Cross-Validation (LITE: train + save folds)
#  - Single train/val split + confusion matrices
#  - Full-train fine-tune
#  - Submissions:
#       * best-val model
#       * full-train model
#       * 2-model ensemble
#       * K-fold ensemble
# ============================================================

# ==== SECTION 0: Install dependencies (Colab) ====
!pip -q install kaggle scikit-learn

# ==== SECTION 1: Colab + Kaggle setup ====
import os, json, getpass, random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# Configure Kaggle API
KAGGLE_USERNAME = input("Enter your Kaggle username: ").strip()
KAGGLE_KEY = getpass.getpass("Enter your Kaggle API key (will be hidden): ").strip()

os.makedirs(Path.home() / ".kaggle", exist_ok=True)
with open(Path.home() / ".kaggle" / "kaggle.json", "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod(Path.home() / ".kaggle" / "kaggle.json", 0o600)

print("Kaggle is configured.")

# ==== SECTION 2: Download competition data ====
COMPETITION = "an2dl2526c1"   # Pirate Pain challenge
assert COMPETITION, "Competition slug is required."

!kaggle competitions files -c $COMPETITION
!kaggle competitions download -c $COMPETITION -p ./. -w
!mkdir -p ./data_extract && unzip -o "./*.zip" -d ./data_extract

print("\nExtracted files:")
!ls -l ./data_extract

TRAIN_FILE = "pirate_pain_train.csv"
LABELS_FILE = "pirate_pain_train_labels.csv"
TEST_FILE  = "pirate_pain_test.csv"
SAMPLE_SUB = "sample_submission.csv"

for fname in [TRAIN_FILE, LABELS_FILE, TEST_FILE, SAMPLE_SUB]:
    path = Path("./data_extract") / fname
    if not path.exists():
        print(f"WARNING: {fname} not found in ./data_extract. Please check filenames.")

# ==== SECTION 3: Data loading & basic checks ====
train_df = pd.read_csv(Path("./data_extract") / TRAIN_FILE)
labels_df = pd.read_csv(Path("./data_extract") / LABELS_FILE)
test_df  = pd.read_csv(Path("./data_extract") / TEST_FILE)

print("train_df shape:", train_df.shape)
print("labels_df shape:", labels_df.shape)
print("test_df shape:",  test_df.shape)

assert "sample_index" in train_df.columns and "time" in train_df.columns
assert "sample_index" in test_df.columns and "time" in test_df.columns
assert "label" in labels_df.columns and "sample_index" in labels_df.columns

train_df = train_df.sort_values(["sample_index", "time"]).reset_index(drop=True)
test_df  = test_df.sort_values(["sample_index", "time"]).reset_index(drop=True)

steps_per_sample = train_df.groupby("sample_index")["time"].nunique()
print("Time steps stats:", steps_per_sample.describe())
T_EXPECTED = 160
if steps_per_sample.nunique() != 1 or steps_per_sample.iloc[0] != T_EXPECTED:
    print("WARNING: Not all samples have the same number of time steps or not equal to 160.")
else:
    print("All samples have 160 time steps.")

# ==== SECTION 4: Static vs dynamic features ====
static_features = ["n_legs", "n_hands", "n_eyes"]

exclude_cols = {"sample_index", "time"}
feature_cols = [c for c in train_df.columns if c not in exclude_cols]
dynamic_features = [c for c in feature_cols if c not in static_features]

print("Static features:", static_features)
print("Dynamic feature count:", len(dynamic_features))
print("Sample dynamic features:", dynamic_features[:10])

for col in static_features:
    print(col, "->", train_df[col].unique())

text2num = {
    "two": 2.0,
    "one+peg_leg": 1.0,
    "one+hook_hand": 1.0,
    "one+eye_patch": 1.0,
    "one": 1.0,
    "zero": 0.0,
}

def convert_static_columns(df, static_cols):
    for col in static_cols:
        df[col] = df[col].map(text2num).fillna(0.0).astype(np.float32)
    return df

train_df = convert_static_columns(train_df, static_features)
test_df  = convert_static_columns(test_df, static_features)

print("\nStatic columns after conversion:")
print(train_df[static_features].head())

# ==== SECTION 5: DataFrame -> (N, T, C_dyn) + static (N, C_static) ====
def df_to_3d_with_static(df, dynamic_cols, static_cols):
    """
    df: rows indexed by (sample_index, time)
    Returns:
        X_dyn: (N, T, C_dyn)
        X_static: (N, C_static)
        sample_ids: (N,)
    """
    groups = df.groupby("sample_index")

    sample_ids = []
    series_list = []   # (T, C_dyn)
    static_list = []   # (C_static,)

    for sid, g in groups:
        g = g.sort_values("time")

        X_dyn = g[dynamic_cols].to_numpy(dtype=np.float32)            # (T, C_dyn)
        X_static = g[static_cols].iloc[0].to_numpy(dtype=np.float32)  # (C_static,)

        sample_ids.append(sid)
        series_list.append(X_dyn)
        static_list.append(X_static)

    X_dyn = np.stack(series_list, axis=0)      # (N, T, C_dyn)
    X_static = np.stack(static_list, axis=0)   # (N, C_static)
    sample_ids = np.array(sample_ids)

    return X_dyn, X_static, sample_ids

X_train_dyn, X_train_static, train_ids = df_to_3d_with_static(
    train_df, dynamic_features, static_features
)
X_test_dyn, X_test_static, test_ids = df_to_3d_with_static(
    test_df, dynamic_features, static_features
)

print("X_train_dyn:", X_train_dyn.shape)      # (N, T, C_dyn)
print("X_train_static:", X_train_static.shape)
print("X_test_dyn:", X_test_dyn.shape)
print("X_test_static:", X_test_static.shape)

# ==== SECTION 6: Label mapping (no_pain=0, low_pain=1, high_pain=2) ====
label_order = ["no_pain", "low_pain", "high_pain"]
label2id = {name: idx for idx, name in enumerate(label_order)}
id2label = {idx: name for idx, name in enumerate(label_order)}

print("Label → ID mapping:", label2id)

labels_map = labels_df.set_index("sample_index")["label"].to_dict()
y_train = np.array([label2id[labels_map[sid]] for sid in train_ids], dtype=np.int64)

print("y_train shape:", y_train.shape)
print("Unique labels:", np.unique(y_train))

# ==== SECTION 7: Class distribution (overall) ====
unique, counts = np.unique(y_train, return_counts=True)
print("\nClass distribution (overall train):")
for cls_id, cnt in zip(unique, counts):
    cls_name = id2label[cls_id]
    frac = cnt / len(y_train)
    print(f"  class {cls_id} ({cls_name}): {cnt} samples ({frac:.2%})")

# ==== SECTION 8: Normalization (dynamic & static separately) ====
dyn_mean = X_train_dyn.mean(axis=(0, 1), keepdims=True)
dyn_std  = X_train_dyn.std(axis=(0, 1), keepdims=True) + 1e-6
X_train_dyn = (X_train_dyn - dyn_mean) / dyn_std
X_test_dyn  = (X_test_dyn  - dyn_mean) / dyn_std

stat_mean = X_train_static.mean(axis=0, keepdims=True)
stat_std  = X_train_static.std(axis=0, keepdims=True) + 1e-6
X_train_static = (X_train_static - stat_mean) / stat_std
X_test_static  = (X_test_static  - stat_mean) / stat_std

print("Static normalized sample:", X_train_static[:5])

# ==== SECTION 9: Dataset class with augmentation ====
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

class TimeSeriesSeqDataset(Dataset):
    def __init__(self, X_dyn, X_stat, y=None, augment=False):
        self.X_dyn = X_dyn          # (N, T, C_dyn)
        self.X_stat = X_stat        # (N, C_static)
        self.y = y                  # (N,) or None
        self.augment = augment

    def __len__(self):
        return self.X_dyn.shape[0]

    def _augment(self, x):
        """
        x: numpy array of shape (T, C_dyn)
        Light time-series augmentations:
          - jitter (Gaussian noise)
          - scaling (global)
          - time-shift (roll along time)
        """
        # Jitter (Gaussian noise)
        if np.random.rand() < 0.7:
            noise = np.random.normal(loc=0.0, scale=0.02, size=x.shape).astype(np.float32)
            x = x + noise

        # Scaling
        if np.random.rand() < 0.3:
            scale = np.random.normal(loc=1.0, scale=0.05)
            x = x * np.float32(scale)

        # Time shift (roll along time axis)
        if np.random.rand() < 0.3:
            max_shift = 5
            shift = np.random.randint(-max_shift, max_shift + 1)
            x = np.roll(x, shift=shift, axis=0)

        return x

    def __getitem__(self, idx):
        x_dyn = self.X_dyn[idx]          # (T, C_dyn)
        x_stat = self.X_stat[idx]        # (C_static,)

        if self.augment:
            x_dyn = self._augment(x_dyn)

        x_dyn = torch.from_numpy(x_dyn).float()
        x_stat = torch.from_numpy(x_stat).float()

        if self.y is None:
            return x_dyn, x_stat

        label = torch.tensor(self.y[idx], dtype=torch.long)
        return x_dyn, x_stat, label

# ==== SECTION 10: Model + train/eval + generic predict_proba ====
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

class AttentionPooling(nn.Module):
    """
    Additive attention over time.
    Input:  x -> (B, T, D)
    Output: context -> (B, D)
    """
    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim, dim)
        self.score = nn.Linear(dim, 1, bias=False)

    def forward(self, x, mask=None):
        # x: (B, T, D)
        h = torch.tanh(self.proj(x))          # (B, T, D)
        scores = self.score(h).squeeze(-1)    # (B, T)

        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))

        weights = torch.softmax(scores, dim=1)          # (B, T)
        context = torch.bmm(weights.unsqueeze(1), x)    # (B, 1, D)
        context = context.squeeze(1)                    # (B, D)
        return context

class CNNGRUWithAttention(nn.Module):
    """
    Dynamic branch:
      - Conv1d stem over time
      - 2-layer BiGRU
      - Multihead self-attention + residual + LayerNorm
      - Triple pooling: mean, max, additive attention
    Static branch:
      - Small MLP on static features
    Then concat and classify.
    """
    def __init__(self, dyn_channels, static_dim, num_classes):
        super().__init__()

        # 1) Conv1d stem: input (B, C_dyn, T)
        self.conv = nn.Sequential(
            nn.Conv1d(
                in_channels=dyn_channels,
                out_channels=64,
                kernel_size=7,
                padding=3,
            ),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.MaxPool1d(kernel_size=2),   # T -> T/2
            nn.Dropout(0.3),
        )

        # 2) BiGRU
        self.gru = nn.GRU(
            input_size=128,
            hidden_size=160,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3,
        )  # output dim = 320
        self.gru_ln = nn.LayerNorm(320)

        # 3) Multi-head self-attention + residual
        self.mha = nn.MultiheadAttention(
            embed_dim=320,
            num_heads=4,
            dropout=0.25,
            batch_first=True,
        )
        self.mha_ln = nn.LayerNorm(320)

        # 4) Additive attention pooling
        self.attn_pool = AttentionPooling(dim=320)

        # 5) Static branch
        if static_dim > 0:
            self.static_mlp = nn.Sequential(
                nn.Linear(static_dim, 8),
                nn.ReLU(),
            )
            static_out_dim = 8
        else:
            self.static_mlp = None
            static_out_dim = 0

        # 6) Classifier
        seq_repr_dim = 320 * 3   # mean + max + attn
        total_dim = seq_repr_dim + static_out_dim

        self.dropout_head = nn.Dropout(0.4)

        self.fc = nn.Sequential(
            nn.Linear(total_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x_dyn, x_stat):
        """
        x_dyn:  (B, T, C_dyn)
        x_stat: (B, C_static)
        """
        # (B, T, C_dyn) -> (B, C_dyn, T)
        x = x_dyn.permute(0, 2, 1)            # (B, C_dyn, T)

        # CNN stem
        x = self.conv(x)                      # (B, 128, T')
        x = x.permute(0, 2, 1)                # (B, T', 128)

        # BiGRU
        out, _ = self.gru(x)                  # (B, T', 320)
        out = self.gru_ln(out)                # (B, T', 320)

        # Multi-head self-attention + residual
        attn_out, _ = self.mha(out, out, out) # (B, T', 320)
        out = self.mha_ln(out + attn_out)     # (B, T', 320)

        # Triple pooling
        mean_pool = out.mean(dim=1)           # (B, 320)
        max_pool, _ = out.max(dim=1)          # (B, 320)
        attn_context = self.attn_pool(out)    # (B, 320)

        x_seq = torch.cat([mean_pool, max_pool, attn_context], dim=1)  # (B, 960)
        x_seq = self.dropout_head(x_seq)

        # Static branch
        if self.static_mlp is not None:
            s = self.static_mlp(x_stat)       # (B, 8)
            z = torch.cat([x_seq, s], dim=1)  # (B, 968)
        else:
            z = x_seq                         # (B, 960)

        logits = self.fc(z)                   # (B, num_classes)
        return logits

dyn_channels = X_train_dyn.shape[-1]
static_dim = X_train_static.shape[-1]
num_classes = len(label_order)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for batch in loader:
        if len(batch) == 3:
            x_dyn, x_stat, y = batch
        else:
            continue

        x_dyn = x_dyn.to(device)
        x_stat = x_stat.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x_dyn, x_stat)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * x_dyn.size(0)

    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_targets = []

    for batch in loader:
        if len(batch) == 3:
            x_dyn, x_stat, y = batch
        else:
            continue

        x_dyn = x_dyn.to(device)
        x_stat = x_stat.to(device)
        y = y.to(device)

        logits = model(x_dyn, x_stat)
        loss = criterion(logits, y)
        total_loss += loss.item() * x_dyn.size(0)

        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.append(preds)
        all_targets.append(y.cpu().numpy())

    total_loss /= len(loader.dataset)
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    macro_f1 = f1_score(all_targets, all_preds, average="macro")
    return total_loss, macro_f1, all_targets, all_preds

@torch.no_grad()
def predict_proba_any(model, loader):
    """
    Works with loaders that yield (x_dyn, x_stat, y) or (x_dyn, x_stat).
    """
    model.eval()
    all_probs = []
    for batch in loader:
        if len(batch) == 3:
            x_dyn, x_stat, _ = batch
        else:
            x_dyn, x_stat = batch

        x_dyn = x_dyn.to(device)
        x_stat = x_stat.to(device)

        logits = model(x_dyn, x_stat)
        probs = F.softmax(logits, dim=1)
        all_probs.append(probs.cpu().numpy())

    return np.concatenate(all_probs, axis=0)

# ==== SECTION 11 (LITE): 5-fold Stratified Cross-Validation (train & save folds) ====
def run_kfold_cv_lite(
    X_dyn, X_static, y,
    n_splits=5,
    n_epochs_cv=60,
    patience_cv=12,
    lr_cv=5e-4
):
    """
    Lite K-fold CV:
      - For each fold:
          * train model with early stopping
          * evaluate best model on that fold's val
          * save best model to 'model_kfold_{fold}.pt'
      - Returns list of val macro-F1 scores (one per fold).
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_scores = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_dyn, y), start=1):
        print(f"\n=== Fold {fold}/{n_splits} ===")

        X_tr_dyn_cv, X_va_dyn_cv = X_dyn[tr_idx], X_dyn[va_idx]
        X_tr_stat_cv, X_va_stat_cv = X_static[tr_idx], X_static[va_idx]
        y_tr_cv, y_va_cv = y[tr_idx], y[va_idx]

        # Datasets
        train_ds_cv = TimeSeriesSeqDataset(X_tr_dyn_cv, X_tr_stat_cv, y_tr_cv, augment=True)
        val_ds_cv   = TimeSeriesSeqDataset(X_va_dyn_cv, X_va_stat_cv, y_va_cv, augment=False)

        # Oversampling for this fold
        class_counts_cv = np.bincount(y_tr_cv)
        class_weights_cv = 1.0 / class_counts_cv
        sample_weights_cv = class_weights_cv[y_tr_cv]
        sample_weights_cv = torch.from_numpy(sample_weights_cv).float()

        sampler_cv = WeightedRandomSampler(
            weights=sample_weights_cv,
            num_samples=len(y_tr_cv),
            replacement=True,
        )

        train_loader_cv = DataLoader(
            train_ds_cv,
            batch_size=256,
            sampler=sampler_cv,
        )
        val_loader_cv = DataLoader(
            val_ds_cv,
            batch_size=256,
            shuffle=False,
        )

        # Model, optimizer, criterion
        model_cv = CNNGRUWithAttention(
            dyn_channels=dyn_channels,
            static_dim=static_dim,
            num_classes=num_classes,
        ).to(device)

        criterion_cv = nn.CrossEntropyLoss(label_smoothing=0.1)
        optimizer_cv = torch.optim.AdamW(model_cv.parameters(), lr=lr_cv, weight_decay=4e-5)

        best_val_f1_cv = -1.0
        best_state_cv = None
        no_improve_cv = 0

        for epoch in range(1, n_epochs_cv + 1):
            train_loss_opt_cv = train_one_epoch(model_cv, train_loader_cv, optimizer_cv, criterion_cv)
            val_loss_cv, val_f1_cv, _, _ = evaluate(model_cv, val_loader_cv, criterion_cv)

            print(
                f"[Fold {fold}] Epoch {epoch:02d} | "
                f"train_loss={train_loss_opt_cv:.4f} | "
                f"val_loss={val_loss_cv:.4f} | "
                f"val_macroF1={val_f1_cv:.4f}"
            )

            if val_f1_cv > best_val_f1_cv + 1e-5:
                best_val_f1_cv = val_f1_cv
                best_state_cv = {k: v.cpu().clone() for k, v in model_cv.state_dict().items()}
                no_improve_cv = 0
            else:
                no_improve_cv += 1
                if no_improve_cv >= patience_cv:
                    print(f"[Fold {fold}] Early stopping.")
                    break

        # Load best and evaluate once more on val (for safety)
        model_cv.load_state_dict(best_state_cv)
        val_loss_best, val_f1_best, val_targets_best, val_preds_best = evaluate(model_cv, val_loader_cv, criterion_cv)
        print(f"[Fold {fold}] Best val macro-F1 (re-eval): {val_f1_best:.4f}")
        fold_scores.append(val_f1_best)

        # Save this fold model
        fold_path = f"model_kfold_{fold}.pt"
        torch.save(model_cv.state_dict(), fold_path)
        print(f"[Fold {fold}] Saved best model to {fold_path}")

    print("\n=== K-fold CV (LITE) results ===")
    for i, score in enumerate(fold_scores, start=1):
        print(f"Fold {i} val macro-F1: {score:.4f}")
    print(f"Mean macro-F1: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")

    return fold_scores

# ---- RUN LITE K-FOLD CV (can be heavy; comment out if needed) ----
cv_scores = run_kfold_cv_lite(
    X_train_dyn, X_train_static, y_train,
    n_splits=5,
    n_epochs_cv=60,
    patience_cv=12,
    lr_cv=5e-4,
)

# ==== SECTION 12: Single stratified train/val split (for final model & plots) ====
skf_single = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tr_idx, va_idx = list(skf_single.split(train_ids, y_train))[0]

X_tr_dyn, X_va_dyn = X_train_dyn[tr_idx], X_train_dyn[va_idx]
X_tr_stat, X_va_stat = X_train_static[tr_idx], X_train_static[va_idx]
y_tr, y_va = y_train[tr_idx], y_train[va_idx]

print("\nClass distribution (train split):")
u_tr, c_tr = np.unique(y_tr, return_counts=True)
for cls_id, cnt in zip(u_tr, c_tr):
    print(f"  class {cls_id} ({id2label[cls_id]}): {cnt} samples ({cnt/len(y_tr):.2%})")

print("\nClass distribution (val split):")
u_va, c_va = np.unique(y_va, return_counts=True)
for cls_id, cnt in zip(u_va, c_va):
    print(f"  class {cls_id} ({id2label[cls_id]}): {cnt} samples ({cnt/len(y_va):.2%})")

BATCH_SIZE = 256

# Train with augmentation=True
train_ds = TimeSeriesSeqDataset(X_tr_dyn, X_tr_stat, y_tr, augment=True)
# Val/test without augmentation
val_ds   = TimeSeriesSeqDataset(X_va_dyn, X_va_stat, y_va, augment=False)
test_ds  = TimeSeriesSeqDataset(X_test_dyn, X_test_static, None, augment=False)

# Oversampling for train split
class_counts_tr = np.bincount(y_tr)
class_weights = 1.0 / class_counts_tr
sample_weights = class_weights[y_tr]
sample_weights = torch.from_numpy(sample_weights).float()

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(y_tr),
    replacement=True,
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=sampler,
)
val_loader   = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
)
test_loader  = DataLoader(
    test_ds,
    batch_size=128,
    shuffle=False,
)

# ==== SECTION 13: Train single model + curves + confusion matrices ====
model = CNNGRUWithAttention(
    dyn_channels=dyn_channels,
    static_dim=static_dim,
    num_classes=num_classes,
).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=4e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3
)

n_epochs = 120
best_val_f1 = -1.0
best_state = None
patience = 12
no_improve = 0

history_train_loss = []
history_val_loss   = []
history_train_f1   = []
history_val_f1     = []

for epoch in range(1, n_epochs + 1):
    train_loss_opt = train_one_epoch(model, train_loader, optimizer, criterion)

    train_loss_eval, train_f1, train_targets, train_preds = evaluate(model, train_loader, criterion)
    val_loss, val_f1, val_targets, val_preds = evaluate(model, val_loader, criterion)

    scheduler.step(val_f1)

    history_train_loss.append(train_loss_opt)
    history_val_loss.append(val_loss)
    history_train_f1.append(train_f1)
    history_val_f1.append(val_f1)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss_opt={train_loss_opt:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"train_macroF1={train_f1:.4f} | "
        f"val_macroF1={val_f1:.4f}"
    )

    if val_f1 > best_val_f1 + 1e-5:
        best_val_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print("Early stopping.")
            break

print("Best val macro-F1 (single split):", best_val_f1)

if best_state is not None:
    model.load_state_dict(best_state)
    torch.save(model.state_dict(), "model_best_val.pt")
    print("Saved best-val model to model_best_val.pt")

# ---- Plots ----
epochs_ran = range(1, len(history_train_loss) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_ran, history_train_loss, label="Train Loss (opt step)")
plt.plot(epochs_ran, history_val_loss,   label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Val Loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs_ran, history_train_f1, label="Train Macro-F1")
plt.plot(epochs_ran, history_val_f1,   label="Val Macro-F1")
plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.title("Train vs Val Macro-F1")
plt.legend()
plt.grid(True)
plt.show()

# ---- Confusion matrices ----
@torch.no_grad()
def get_preds_and_targets(model, loader):
    model.eval()
    all_preds = []
    all_targets = []
    for batch in loader:
        if len(batch) == 3:
            x_dyn, x_stat, y = batch
        else:
            continue
        x_dyn = x_dyn.to(device)
        x_stat = x_stat.to(device)
        y = y.to(device)

        logits = model(x_dyn, x_stat)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.append(preds)
        all_targets.append(y.cpu().numpy())
    return np.concatenate(all_targets), np.concatenate(all_preds)

best_model = CNNGRUWithAttention(
    dyn_channels=dyn_channels,
    static_dim=static_dim,
    num_classes=num_classes,
).to(device)
best_model.load_state_dict(torch.load("model_best_val.pt", map_location=device))

train_targets_cm, train_preds_cm = get_preds_and_targets(best_model, train_loader)
cm_train = confusion_matrix(train_targets_cm, train_preds_cm, labels=[0, 1, 2])

plt.figure(figsize=(5, 4))
disp_train = ConfusionMatrixDisplay(
    confusion_matrix=cm_train,
    display_labels=[id2label[i] for i in [0, 1, 2]]
)
disp_train.plot(values_format="d", cmap="Blues", colorbar=False)
plt.title("Confusion Matrix - TRAIN (best model)")
plt.show()

val_targets_cm, val_preds_cm = get_preds_and_targets(best_model, val_loader)
cm_val = confusion_matrix(val_targets_cm, val_preds_cm, labels=[0, 1, 2])

plt.figure(figsize=(5, 4))
disp_val = ConfusionMatrixDisplay(
    confusion_matrix=cm_val,
    display_labels=[id2label[i] for i in [0, 1, 2]]
)
disp_val.plot(values_format="d", cmap="Blues", colorbar=False)
plt.title("Confusion Matrix - VAL (best model)")
plt.show()

# ==== SECTION 14: Full-train fine-tune & save full model ====
full_ds = TimeSeriesSeqDataset(X_train_dyn, X_train_static, y_train, augment=True)

class_counts_full = np.bincount(y_train)
class_weights_full = 1.0 / class_counts_full
sample_weights_full = class_weights_full[y_train]
sample_weights_full = torch.from_numpy(sample_weights_full).float()

sampler_full = WeightedRandomSampler(
    weights=sample_weights_full,
    num_samples=len(y_train),
    replacement=True,
)

full_loader = DataLoader(
    full_ds,
    batch_size=BATCH_SIZE,
    sampler=sampler_full,
)

model_full = CNNGRUWithAttention(
    dyn_channels=dyn_channels,
    static_dim=static_dim,
    num_classes=num_classes,
).to(device)

state_best = torch.load("model_best_val.pt", map_location=device)
model_full.load_state_dict(state_best)

optimizer_full = torch.optim.AdamW(model_full.parameters(), lr=3e-4, weight_decay=4e-5)
criterion_full = nn.CrossEntropyLoss(label_smoothing=0.1)

FINE_EPOCHS = 5
for epoch in range(1, FINE_EPOCHS + 1):
    loss_full = train_one_epoch(model_full, full_loader, optimizer_full, criterion_full)
    print(f"[Full train] Epoch {epoch:02d} | loss={loss_full:.4f}")

torch.save(model_full.state_dict(), "model_full_train.pt")
print("Saved full-train model to model_full_train.pt")

# ==== SECTION 15: Inference on TEST + VAL metrics + all ensembles ====
print("\n=== Inference on TEST + VAL metrics ===")

y_va_np = y_va.copy()  # for convenience

# ---- Helper for ensemble CE loss on val ----
def ce_loss_from_probs(probs, y_true):
    # probs: (N, num_classes), y_true: (N,)
    eps = 1e-12
    p_true = probs[np.arange(len(y_true)), y_true]
    return -np.log(p_true + eps).mean()

# (A) Best-val model
print("\n--- Best-val model ---")
model_best = CNNGRUWithAttention(
    dyn_channels=dyn_channels,
    static_dim=static_dim,
    num_classes=num_classes,
).to(device)
model_best.load_state_dict(torch.load("model_best_val.pt", map_location=device))

# VAL metrics
val_loss_best, val_f1_best, _, _ = evaluate(model_best, val_loader, criterion)
print(f"[VAL] best-val model: loss={val_loss_best:.4f}, macroF1={val_f1_best:.4f}")

# PROBS on VAL & TEST
val_probs_best  = predict_proba_any(model_best, val_loader)
test_probs_best = predict_proba_any(model_best, test_loader)

# (B) Full-train model
print("\n--- Full-train model ---")
model_full_load = CNNGRUWithAttention(
    dyn_channels=dyn_channels,
    static_dim=static_dim,
    num_classes=num_classes,
).to(device)
model_full_load.load_state_dict(torch.load("model_full_train.pt", map_location=device))

# VAL metrics
val_loss_full, val_f1_full, _, _ = evaluate(model_full_load, val_loader, criterion)
print(f"[VAL] full-train model: loss={val_loss_full:.4f}, macroF1={val_f1_full:.4f}")

# PROBS on VAL & TEST
val_probs_full  = predict_proba_any(model_full_load, val_loader)
test_probs_full = predict_proba_any(model_full_load, test_loader)

# (C) 2-model ensemble (best-val + full-train)
print("\n--- Ensemble: best-val + full-train ---")
val_probs_2ens  = 0.5 * val_probs_best  + 0.5 * val_probs_full
test_probs_2ens = 0.5 * test_probs_best + 0.5 * test_probs_full

val_preds_2ens = val_probs_2ens.argmax(axis=1)
val_f1_2ens    = f1_score(y_va_np, val_preds_2ens, average="macro")
val_loss_2ens  = ce_loss_from_probs(val_probs_2ens, y_va_np)

print(f"[VAL] 2-model ensemble: loss={val_loss_2ens:.4f}, macroF1={val_f1_2ens:.4f}")

# Submission for 2-model ensemble
test_pred_ids_2ens    = test_probs_2ens.argmax(axis=1)
test_pred_labels_2ens = [id2label[i] for i in test_pred_ids_2ens]

submission_2ens = pd.DataFrame({
    "sample_index": test_ids,
    "label": test_pred_labels_2ens
}).sort_values("sample_index")
submission_2ens.to_csv("submission_ensemble_2models.csv", index=False)
print("Saved submission_ensemble_2models.csv")

# (D) K-fold ensemble (models from run_kfold_cv_lite) on VAL & TEST
print("\n--- K-fold ensemble (models from run_kfold_cv_lite) ---")
kfold_probs_val_list  = []
kfold_probs_test_list = []
n_splits_k = 5

for fold in range(1, n_splits_k + 1):
    path_fold = f"model_kfold_{fold}.pt"
    if not os.path.exists(path_fold):
        print(f"WARNING: {path_fold} not found, skipping this fold.")
        continue

    m_fold = CNNGRUWithAttention(
        dyn_channels=dyn_channels,
        static_dim=static_dim,
        num_classes=num_classes,
    ).to(device)
    m_fold.load_state_dict(torch.load(path_fold, map_location=device))

    probs_val_fold  = predict_proba_any(m_fold, val_loader)
    probs_test_fold = predict_proba_any(m_fold, test_loader)

    kfold_probs_val_list.append(probs_val_fold)
    kfold_probs_test_list.append(probs_test_fold)

if len(kfold_probs_val_list) > 0:
    kfold_probs_val_stack  = np.stack(kfold_probs_val_list,  axis=0)  # (K, N_val,  num_classes)
    kfold_probs_test_stack = np.stack(kfold_probs_test_list, axis=0)  # (K, N_test, num_classes)

    val_probs_kfold_ens  = kfold_probs_val_stack.mean(axis=0)
    test_probs_kfold_ens = kfold_probs_test_stack.mean(axis=0)

    val_preds_kfold = val_probs_kfold_ens.argmax(axis=1)
    val_f1_kfold    = f1_score(y_va_np, val_preds_kfold, average="macro")
    val_loss_kfold  = ce_loss_from_probs(val_probs_kfold_ens, y_va_np)

    print(f"[VAL] K-fold ensemble: loss={val_loss_kfold:.4f}, macroF1={val_f1_kfold:.4f}")

    # Submission for K-fold ensemble
    test_pred_ids_kens    = test_probs_kfold_ens.argmax(axis=1)
    test_pred_labels_kens = [id2label[i] for i in test_pred_ids_kens]

    submission_kfold_ens = pd.DataFrame({
        "sample_index": test_ids,
        "label": test_pred_labels_kens
    }).sort_values("sample_index")
    submission_kfold_ens.to_csv("submission_kfold_ensemble.csv", index=False)
    print("Saved submission_kfold_ensemble.csv")

    # (E) Final ensemble: combine 2-model ensemble + K-fold ensemble
    print("\n--- FINAL ensemble: (2-model ensemble) + (K-fold ensemble) ---")
    val_probs_final  = 0.5 * val_probs_2ens  + 0.5 * val_probs_kfold_ens
    test_probs_final = 0.5 * test_probs_2ens + 0.5 * test_probs_kfold_ens

    val_preds_final = val_probs_final.argmax(axis=1)
    val_f1_final    = f1_score(y_va_np, val_preds_final, average="macro")
    val_loss_final  = ce_loss_from_probs(val_probs_final, y_va_np)

    print(f"[VAL] FINAL ensemble: loss={val_loss_final:.4f}, macroF1={val_f1_final:.4f}")

    test_pred_ids_final    = test_probs_final.argmax(axis=1)
    test_pred_labels_final = [id2label[i] for i in test_pred_ids_final]

    submission_final = pd.DataFrame({
        "sample_index": test_ids,
        "label": test_pred_labels_final
    }).sort_values("sample_index")
    submission_final.to_csv("submission_final_ensemble.csv", index=False)
    print("Saved submission_final_ensemble.csv")

else:
    print("No k-fold models found; skipping K-fold and final ensemble.")


In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# Pirate Pain – Preprocessing + CNN + BiGRU + MHA + Attn + Static
# With CLASS UNDERSAMPLING, training curves, F1 on train/val, confusion matrices,
# best-val model + full-train fine-tune + two submissions
# ============================================================

# ==== SECTION 0: Install dependencies (Colab) ====
!pip -q install kaggle scikit-learn

# ==== SECTION 1: Colab + Kaggle setup ====
import os, json, getpass, random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# Configure Kaggle API
KAGGLE_USERNAME = input("Enter your Kaggle username: ").strip()
KAGGLE_KEY = getpass.getpass("Enter your Kaggle API key (will be hidden): ").strip()

os.makedirs(Path.home() / ".kaggle", exist_ok=True)
with open(Path.home() / ".kaggle" / "kaggle.json", "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod(Path.home() / ".kaggle" / "kaggle.json", 0o600)

print("Kaggle is configured.")

# ==== SECTION 2: Download competition data ====
COMPETITION = "an2dl2526c1"   # Pirate Pain challenge
assert COMPETITION, "Competition slug is required."

!kaggle competitions files -c $COMPETITION
!kaggle competitions download -c $COMPETITION -p ./. -w
!mkdir -p ./data_extract && unzip -o "./*.zip" -d ./data_extract

print("\nExtracted files:")
!ls -l ./data_extract

TRAIN_FILE = "pirate_pain_train.csv"
LABELS_FILE = "pirate_pain_train_labels.csv"
TEST_FILE  = "pirate_pain_test.csv"
SAMPLE_SUB = "sample_submission.csv"

for fname in [TRAIN_FILE, LABELS_FILE, TEST_FILE, SAMPLE_SUB]:
    path = Path("./data_extract") / fname
    if not path.exists():
        print(f"WARNING: {fname} not found in ./data_extract. Please check filenames.")

# ==== SECTION 3: Data loading & basic checks ====
train_df = pd.read_csv(Path("./data_extract") / TRAIN_FILE)
labels_df = pd.read_csv(Path("./data_extract") / LABELS_FILE)
test_df  = pd.read_csv(Path("./data_extract") / TEST_FILE)

print("train_df shape:", train_df.shape)
print("labels_df shape:", labels_df.shape)
print("test_df shape:",  test_df.shape)

assert "sample_index" in train_df.columns and "time" in train_df.columns
assert "sample_index" in test_df.columns and "time" in test_df.columns
assert "label" in labels_df.columns and "sample_index" in labels_df.columns

train_df = train_df.sort_values(["sample_index", "time"]).reset_index(drop=True)
test_df  = test_df.sort_values(["sample_index", "time"]).reset_index(drop=True)

steps_per_sample = train_df.groupby("sample_index")["time"].nunique()
print("Time steps stats:", steps_per_sample.describe())
T_EXPECTED = 160
if steps_per_sample.nunique() != 1 or steps_per_sample.iloc[0] != T_EXPECTED:
    print("WARNING: Not all samples have the same number of time steps or not equal to 160.")
else:
    print("All samples have 160 time steps.")

# ==== SECTION 4: Static vs dynamic features (your style) ====
static_features = ["n_legs", "n_hands", "n_eyes"]

exclude_cols = {"sample_index", "time"}
feature_cols = [c for c in train_df.columns if c not in exclude_cols]

dynamic_features = [c for c in feature_cols if c not in static_features]

print("Static features:", static_features)
print("Dynamic feature count:", len(dynamic_features))
print("Sample dynamic features:", dynamic_features[:10])

for col in static_features:
    print(col, "->", train_df[col].unique())

text2num = {
    "two": 2.0,
    "one+peg_leg": 1.0,
    "one+hook_hand": 1.0,
    "one+eye_patch": 1.0,
    "one": 1.0,
    "zero": 0.0,
}

def convert_static_columns(df, static_cols):
    for col in static_cols:
        df[col] = df[col].map(text2num).fillna(0.0).astype(np.float32)
    return df

train_df = convert_static_columns(train_df, static_features)
test_df  = convert_static_columns(test_df, static_features)

print("\nStatic columns after conversion:")
print(train_df[static_features].head())

# ==== SECTION 5: DataFrame -> (N, T, C_dyn) + static (N, C_static) ====
def df_to_3d_with_static(df, dynamic_cols, static_cols):
    """
    df: rows indexed by (sample_index, time)
    Returns:
        X_dyn: (N, T, C_dyn)
        X_static: (N, C_static)
        sample_ids: (N,)
    """
    groups = df.groupby("sample_index")

    sample_ids = []
    series_list = []   # (T, C_dyn)
    static_list = []   # (C_static,)

    for sid, g in groups:
        g = g.sort_values("time")

        X_dyn = g[dynamic_cols].to_numpy(dtype=np.float32)            # (T, C_dyn)
        X_static = g[static_cols].iloc[0].to_numpy(dtype=np.float32)  # (C_static,)

        sample_ids.append(sid)
        series_list.append(X_dyn)
        static_list.append(X_static)

    X_dyn = np.stack(series_list, axis=0)      # (N, T, C_dyn)
    X_static = np.stack(static_list, axis=0)   # (N, C_static)
    sample_ids = np.array(sample_ids)

    return X_dyn, X_static, sample_ids

X_train_dyn, X_train_static, train_ids = df_to_3d_with_static(
    train_df, dynamic_features, static_features
)
X_test_dyn, X_test_static, test_ids = df_to_3d_with_static(
    test_df, dynamic_features, static_features
)

print("X_train_dyn:", X_train_dyn.shape)      # (N, T, C_dyn)
print("X_train_static:", X_train_static.shape)
print("X_test_dyn:", X_test_dyn.shape)
print("X_test_static:", X_test_static.shape)

# ==== SECTION 6: Label mapping (no_pain=0, low_pain=1, high_pain=2) ====
label_order = ["no_pain", "low_pain", "high_pain"]
label2id = {name: idx for idx, name in enumerate(label_order)}
id2label = {idx: name for idx, name in enumerate(label_order)}

print("Label → ID mapping:", label2id)

labels_map = labels_df.set_index("sample_index")["label"].to_dict()
y_train = np.array([label2id[labels_map[sid]] for sid in train_ids], dtype=np.int64)

print("y_train shape:", y_train.shape)
print("Unique labels:", np.unique(y_train))

# ==== SECTION 7: Class distribution (overall) ====
unique, counts = np.unique(y_train, return_counts=True)
print("\nClass distribution (overall train):")
for cls_id, cnt in zip(unique, counts):
    cls_name = id2label[cls_id]
    frac = cnt / len(y_train)
    print(f"  class {cls_id} ({cls_name}): {cnt} samples ({frac:.2%})")

# ==== SECTION 8: Normalization (dynamic & static separately) ====
dyn_mean = X_train_dyn.mean(axis=(0, 1), keepdims=True)
dyn_std  = X_train_dyn.std(axis=(0, 1), keepdims=True) + 1e-6
X_train_dyn = (X_train_dyn - dyn_mean) / dyn_std
X_test_dyn  = (X_test_dyn  - dyn_mean) / dyn_std

stat_mean = X_train_static.mean(axis=0, keepdims=True)
stat_std  = X_train_static.std(axis=0, keepdims=True) + 1e-6
X_train_static = (X_train_static - stat_mean) / stat_std
X_test_static  = (X_test_static  - stat_mean) / stat_std

print("Static normalized sample:", X_train_static[:5])

# ==== SECTION 9: Stratified train/val split ====
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tr_idx, va_idx = list(skf.split(train_ids, y_train))[0]

X_tr_dyn, X_va_dyn = X_train_dyn[tr_idx], X_train_dyn[va_idx]
X_tr_stat, X_va_stat = X_train_static[tr_idx], X_train_static[va_idx]
y_tr, y_va = y_train[tr_idx], y_train[va_idx]

print("\nClass distribution (train split BEFORE undersampling):")
u_tr, c_tr = np.unique(y_tr, return_counts=True)
for cls_id, cnt in zip(u_tr, c_tr):
    print(f"  class {cls_id} ({id2label[cls_id]}): {cnt} samples ({cnt/len(y_tr):.2%})")

print("\nClass distribution (val split):")
u_va, c_va = np.unique(y_va, return_counts=True)
for cls_id, cnt in zip(u_va, c_va):
    print(f"  class {cls_id} ({id2label[cls_id]}): {cnt} samples ({cnt/len(y_va):.2%})")

# ==== SECTION 10: Random undersampling on TRAIN split ====
rng = np.random.default_rng(42)
classes = np.unique(y_tr)
class_counts = np.bincount(y_tr)
min_count = class_counts[class_counts > 0].min()   # smallest non-zero count

balanced_indices = []
for c in classes:
    idx_c = np.where(y_tr == c)[0]
    chosen = rng.choice(idx_c, size=min_count, replace=False)
    balanced_indices.append(chosen)

balanced_indices = np.concatenate(balanced_indices)
rng.shuffle(balanced_indices)

X_tr_dyn = X_tr_dyn[balanced_indices]
X_tr_stat = X_tr_stat[balanced_indices]
y_tr = y_tr[balanced_indices]

print("\nClass distribution (train split AFTER undersampling):")
u_tr2, c_tr2 = np.unique(y_tr, return_counts=True)
for cls_id, cnt in zip(u_tr2, c_tr2):
    print(f"  class {cls_id} ({id2label[cls_id]}): {cnt} samples ({cnt/len(y_tr):.2%})")

# ==== SECTION 11: Dataset & Dataloaders (NO augmentation) ====
from torch.utils.data import Dataset, DataLoader

class TimeSeriesSeqDataset(Dataset):
    def __init__(self, X_dyn, X_stat, y=None, augment=False):
        self.X_dyn = X_dyn          # (N, T, C_dyn)
        self.X_stat = X_stat        # (N, C_static)
        self.y = y                  # (N,) or None
        self.augment = augment      # will be False here

    def __len__(self):
        return self.X_dyn.shape[0]

    def _augment(self, x):
        # not used (augment=False)
        return x

    def __getitem__(self, idx):
        x_dyn = self.X_dyn[idx]          # (T, C_dyn)
        x_stat = self.X_stat[idx]        # (C_static,)

        if self.augment:
            x_dyn = self._augment(x_dyn)

        x_dyn = torch.from_numpy(x_dyn).float()
        x_stat = torch.from_numpy(x_stat).float()

        if self.y is None:
            return x_dyn, x_stat

        label = torch.tensor(self.y[idx], dtype=torch.long)
        return x_dyn, x_stat, label

BATCH_SIZE = 256

train_ds = TimeSeriesSeqDataset(X_tr_dyn, X_tr_stat, y_tr, augment=False)
val_ds   = TimeSeriesSeqDataset(X_va_dyn, X_va_stat, y_va, augment=False)
test_ds  = TimeSeriesSeqDataset(X_test_dyn, X_test_static, None, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=128, shuffle=False)

# ==== SECTION 12: Model – CNN + BiGRU + Multi-Head Attention + Additive Attention + Static ====
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

class AttentionPooling(nn.Module):
    """
    Additive attention over time.
    Input:  x -> (B, T, D)
    Output: context -> (B, D)
    """
    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim, dim)
        self.score = nn.Linear(dim, 1, bias=False)

    def forward(self, x, mask=None):
        # x: (B, T, D)
        h = torch.tanh(self.proj(x))          # (B, T, D)
        scores = self.score(h).squeeze(-1)    # (B, T)

        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))

        weights = torch.softmax(scores, dim=1)          # (B, T)
        context = torch.bmm(weights.unsqueeze(1), x)    # (B, 1, D)
        context = context.squeeze(1)                    # (B, D)
        return context

class CNNGRUWithAttention(nn.Module):
    """
    Dynamic branch:
      - Conv1d stem over time
      - 2-layer BiGRU
      - Multihead self-attention + residual + LayerNorm
      - Triple pooling: mean, max, additive attention
    Static branch:
      - Small MLP on static features
    Then concat and classify.
    """
    def __init__(self, dyn_channels, static_dim, num_classes):
        super().__init__()

        # 1) Conv1d stem: input (B, C_dyn, T)
        self.conv = nn.Sequential(
            nn.Conv1d(
                in_channels=dyn_channels,
                out_channels=64,
                kernel_size=7,
                padding=3,
            ),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.MaxPool1d(kernel_size=2),   # T -> T/2
            nn.Dropout(0.3),
        )

        # 2) BiGRU
        self.gru = nn.GRU(
            input_size=128,
            hidden_size=160,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3,
        )  # output dim = 320
        self.gru_ln = nn.LayerNorm(320)

        # 3) Multi-head self-attention + residual
        self.mha = nn.MultiheadAttention(
            embed_dim=320,
            num_heads=4,
            dropout=0.25,
            batch_first=True,
        )
        self.mha_ln = nn.LayerNorm(320)

        # 4) Additive attention pooling
        self.attn_pool = AttentionPooling(dim=320)

        # 5) Static branch
        if static_dim > 0:
            self.static_mlp = nn.Sequential(
                nn.Linear(static_dim, 8),
                nn.ReLU(),
            )
            static_out_dim = 8
        else:
            self.static_mlp = None
            static_out_dim = 0

        # 6) Classifier
        seq_repr_dim = 320 * 3   # mean + max + attn
        total_dim = seq_repr_dim + static_out_dim

        self.dropout_head = nn.Dropout(0.4)

        self.fc = nn.Sequential(
            nn.Linear(total_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x_dyn, x_stat):
        """
        x_dyn:  (B, T, C_dyn)
        x_stat: (B, C_static)
        """
        # (B, T, C_dyn) -> (B, C_dyn, T)
        x = x_dyn.permute(0, 2, 1)            # (B, C_dyn, T)

        # CNN stem
        x = self.conv(x)                      # (B, 128, T')
        x = x.permute(0, 2, 1)                # (B, T', 128)

        # BiGRU
        out, _ = self.gru(x)                  # (B, T', 320)
        out = self.gru_ln(out)                # (B, T', 320)

        # Multi-head self-attention + residual
        attn_out, _ = self.mha(out, out, out) # (B, T', 320)
        out = self.mha_ln(out + attn_out)     # (B, T', 320)

        # Triple pooling
        mean_pool = out.mean(dim=1)           # (B, 320)
        max_pool, _ = out.max(dim=1)          # (B, 320)
        attn_context = self.attn_pool(out)    # (B, 320)

        x_seq = torch.cat([mean_pool, max_pool, attn_context], dim=1)  # (B, 960)
        x_seq = self.dropout_head(x_seq)

        # Static branch
        if self.static_mlp is not None:
            s = self.static_mlp(x_stat)       # (B, 8)
            z = torch.cat([x_seq, s], dim=1)  # (B, 968)
        else:
            z = x_seq                         # (B, 960)

        logits = self.fc(z)                   # (B, num_classes)
        return logits

dyn_channels = X_train_dyn.shape[-1]
static_dim = X_train_static.shape[-1]
num_classes = len(label_order)

model = CNNGRUWithAttention(
    dyn_channels=dyn_channels,
    static_dim=static_dim,
    num_classes=num_classes,
).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=4e-5)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3
)

# ==== SECTION 13: Train with early stopping + history & plots + confusion matrices ====
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for x_dyn, x_stat, y in loader:
        x_dyn = x_dyn.to(device)
        x_stat = x_stat.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x_dyn, x_stat)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * x_dyn.size(0)

    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_targets = []

    for x_dyn, x_stat, y in loader:
        x_dyn = x_dyn.to(device)
        x_stat = x_stat.to(device)
        y = y.to(device)

        logits = model(x_dyn, x_stat)
        loss = criterion(logits, y)
        total_loss += loss.item() * x_dyn.size(0)

        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.append(preds)
        all_targets.append(y.cpu().numpy())

    total_loss /= len(loader.dataset)
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    macro_f1 = f1_score(all_targets, all_preds, average="macro")
    return total_loss, macro_f1, all_targets, all_preds

n_epochs = 120
best_val_f1 = -1.0
best_state = None
patience = 12
no_improve = 0

history_train_loss = []
history_val_loss   = []
history_train_f1   = []
history_val_f1     = []

for epoch in range(1, n_epochs + 1):
    # Train one epoch (optimization loss)
    train_loss_opt = train_one_epoch(model, train_loader, optimizer, criterion)

    # Evaluate on train and val (to get F1 for both)
    train_loss_eval, train_f1, train_targets, train_preds = evaluate(model, train_loader, criterion)
    val_loss, val_f1, val_targets, val_preds = evaluate(model, val_loader, criterion)

    scheduler.step(val_f1)

    history_train_loss.append(train_loss_opt)
    history_val_loss.append(val_loss)
    history_train_f1.append(train_f1)
    history_val_f1.append(val_f1)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss_opt={train_loss_opt:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"train_macroF1={train_f1:.4f} | "
        f"val_macroF1={val_f1:.4f}"
    )

    # Early stopping based on val F1
    if val_f1 > best_val_f1 + 1e-5:
        best_val_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print("Early stopping.")
            break

print("Best val macro-F1:", best_val_f1)

# Load and save best-val model
if best_state is not None:
    model.load_state_dict(best_state)
    torch.save(model.state_dict(), "model_best_val.pt")
    print("Saved best-val model to model_best_val.pt")

# ---- 13-B: Plot training curves (loss + F1) ----
epochs_ran = range(1, len(history_train_loss) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_ran, history_train_loss, label="Train Loss (opt step)")
plt.plot(epochs_ran, history_val_loss,   label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Val Loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs_ran, history_train_f1, label="Train Macro-F1")
plt.plot(epochs_ran, history_val_f1,   label="Val Macro-F1")
plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.title("Train vs Val Macro-F1")
plt.legend()
plt.grid(True)
plt.show()

# ---- 13-C: Confusion matrices for TRAIN and VAL using best model ----
@torch.no_grad()
def get_preds_and_targets(model, loader):
    model.eval()
    all_preds = []
    all_targets = []
    for x_dyn, x_stat, y in loader:
        x_dyn = x_dyn.to(device)
        x_stat = x_stat.to(device)
        y = y.to(device)

        logits = model(x_dyn, x_stat)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.append(preds)
        all_targets.append(y.cpu().numpy())
    return np.concatenate(all_targets), np.concatenate(all_preds)

best_model = CNNGRUWithAttention(
    dyn_channels=dyn_channels,
    static_dim=static_dim,
    num_classes=num_classes,
).to(device)
best_model.load_state_dict(torch.load("model_best_val.pt", map_location=device))

# Train confusion matrix
train_targets_cm, train_preds_cm = get_preds_and_targets(best_model, train_loader)
cm_train = confusion_matrix(train_targets_cm, train_preds_cm, labels=[0, 1, 2])

plt.figure(figsize=(5, 4))
disp_train = ConfusionMatrixDisplay(
    confusion_matrix=cm_train,
    display_labels=[id2label[i] for i in [0, 1, 2]]
)
disp_train.plot(values_format="d", cmap="Blues", colorbar=False)
plt.title("Confusion Matrix - TRAIN (best model)")
plt.show()

# Val confusion matrix
val_targets_cm, val_preds_cm = get_preds_and_targets(best_model, val_loader)
cm_val = confusion_matrix(val_targets_cm, val_preds_cm, labels=[0, 1, 2])

plt.figure(figsize=(5, 4))
disp_val = ConfusionMatrixDisplay(
    confusion_matrix=cm_val,
    display_labels=[id2label[i] for i in [0, 1, 2]]
)
disp_val.plot(values_format="d", cmap="Blues", colorbar=False)
plt.title("Confusion Matrix - VAL (best model)")
plt.show()

# ==== SECTION 14: Full-train (with undersampling on full train) & save full model ====
rng_full = np.random.default_rng(123)
classes_full = np.unique(y_train)
counts_full = np.bincount(y_train)
min_count_full = counts_full[counts_full > 0].min()

balanced_full_indices = []
for c in classes_full:
    idx_c = np.where(y_train == c)[0]
    chosen = rng_full.choice(idx_c, size=min_count_full, replace=False)
    balanced_full_indices.append(chosen)

balanced_full_indices = np.concatenate(balanced_full_indices)
rng_full.shuffle(balanced_full_indices)

X_full_dyn = X_train_dyn[balanced_full_indices]
X_full_stat = X_train_static[balanced_full_indices]
y_full = y_train[balanced_full_indices]

print("\nClass distribution (FULL train AFTER undersampling):")
u_f, c_f = np.unique(y_full, return_counts=True)
for cls_id, cnt in zip(u_f, c_f):
    print(f"  class {cls_id} ({id2label[cls_id]}): {cnt} samples ({cnt/len(y_full):.2%})")

full_ds = TimeSeriesSeqDataset(X_full_dyn, X_full_stat, y_full, augment=False)
full_loader = DataLoader(full_ds, batch_size=BATCH_SIZE, shuffle=True)

model_full = CNNGRUWithAttention(
    dyn_channels=dyn_channels,
    static_dim=static_dim,
    num_classes=num_classes,
).to(device)

state_best = torch.load("model_best_val.pt", map_location=device)
model_full.load_state_dict(state_best)

optimizer_full = torch.optim.AdamW(model_full.parameters(), lr=3e-4, weight_decay=4e-5)

FINE_EPOCHS = 5
for epoch in range(1, FINE_EPOCHS + 1):
    loss_full = train_one_epoch(model_full, full_loader, optimizer_full, criterion)
    print(f"[Full train] Epoch {epoch:02d} | loss={loss_full:.4f}")

torch.save(model_full.state_dict(), "model_full_train.pt")
print("Saved full-train model to model_full_train.pt")

# ==== SECTION 15: Inference & submissions for both models ====
@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    all_probs = []
    for x_dyn, x_stat in loader:
        x_dyn = x_dyn.to(device)
        x_stat = x_stat.to(device)

        logits = model(x_dyn, x_stat)
        probs = F.softmax(logits, dim=1)
        all_probs.append(probs.cpu().numpy())
    return np.concatenate(all_probs, axis=0)

# 1) Best-val model submission
print("\n=== Inference with best-val model ===")
model_best = CNNGRUWithAttention(
    dyn_channels=dyn_channels,
    static_dim=static_dim,
    num_classes=num_classes,
).to(device)
model_best.load_state_dict(torch.load("model_best_val.pt", map_location=device))

test_probs_best = predict_proba(model_best, test_loader)
test_pred_ids_best = test_probs_best.argmax(axis=1)
test_pred_labels_best = [id2label[i] for i in test_pred_ids_best]

submission_best = pd.DataFrame({
    "sample_index": test_ids,
    "label": test_pred_labels_best
}).sort_values("sample_index")

submission_best.to_csv("submission_best_val.csv", index=False)
print(submission_best.head())
print("Saved submission_best_val.csv")

# 2) Full-train model submission
print("\n=== Inference with full-train model ===")
model_full_load = CNNGRUWithAttention(
    dyn_channels=dyn_channels,
    static_dim=static_dim,
    num_classes=num_classes,
).to(device)
model_full_load.load_state_dict(torch.load("model_full_train.pt", map_location=device))

test_probs_full = predict_proba(model_full_load, test_loader)
test_pred_ids_full = test_probs_full.argmax(axis=1)
test_pred_labels_full = [id2label[i] for i in test_pred_ids_full]

submission_full = pd.DataFrame({
    "sample_index": test_ids,
    "label": test_pred_labels_full
}).sort_values("sample_index")

submission_full.to_csv("submission_full_train.csv", index=False)
print(submission_full.head())
print("Saved submission_full_train.csv")


In [ ]:
# ============================================================
# AN2DL Pirates Pain - Full pipeline (Kaggle download included)
# CNN + BiGRU + Attention + Static features + Oversampling
# Optional Focal Loss + Ensembles + Confusion Matrices
# ============================================================

# =======================
# SECTION 0: Install & Kaggle download
# =======================
!pip -q install scikit-learn kaggle

import os, json, getpass
from pathlib import Path

# --- Configure Kaggle credentials (run once) ---
KAGGLE_USERNAME = input("Enter your Kaggle username: ").strip()
KAGGLE_KEY = getpass.getpass("Enter your Kaggle API key (hidden): ").strip()

os.makedirs(Path.home()/".kaggle", exist_ok=True)
with open(Path.home()/".kaggle"/"kaggle.json", "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod(Path.home()/".kaggle"/"kaggle.json", 0o600)

print("Kaggle configured.")

COMPETITION = "an2dl2526c1"

!kaggle competitions files -c $COMPETITION
!kaggle competitions download -c $COMPETITION -p . -w
!mkdir -p ./data_extract && unzip -o "./*.zip" -d ./data_extract

print("\nExtracted files:")
!ls -l ./data_extract

# =======================
# SECTION 1: Imports & Global configs
# =======================
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

from pathlib import Path

SEED = 42
BATCH_SIZE = 64

N_EPOCHS_MAIN = 70      # train/val
PATIENCE_MAIN = 15

N_EPOCHS_CV = 70        # k-fold
PATIENCE_CV = 15

FINE_EPOCHS_FULL = 20   # full-train fine-tune

LR_MAIN  = 5e-4
LR_CV    = 5e-4
LR_FULL  = 5e-4

# ==== Focal Loss config ====
USE_FOCAL_LOSS       = True   # True to use FocalLoss, False for CrossEntropy
FOCAL_USE_ALPHA      = False   # if True, use class-based alpha (1/freq)
FOCAL_GAMMA          = 1
FOCAL_LABEL_SMOOTH   = 0.0     # usually 0.0 for focal

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# =======================
# SECTION 2: Reproducibility
# =======================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# =======================
# SECTION 3: Load CSVs from ./data_extract
# =======================
DATA_DIR = Path("./data_extract")

TRAIN_FILE  = DATA_DIR / "pirate_pain_train.csv"
LABELS_FILE = DATA_DIR / "pirate_pain_train_labels.csv"
TEST_FILE   = DATA_DIR / "pirate_pain_test.csv"

assert TRAIN_FILE.exists(), f"{TRAIN_FILE} not found"
assert LABELS_FILE.exists(), f"{LABELS_FILE} not found"
assert TEST_FILE.exists(), f"{TEST_FILE} not found"

train_df  = pd.read_csv(TRAIN_FILE)
labels_df = pd.read_csv(LABELS_FILE)
test_df   = pd.read_csv(TEST_FILE)

assert "sample_index" in train_df.columns and "time" in train_df.columns
assert "sample_index" in test_df.columns and "time" in test_df.columns
assert "label" in labels_df.columns and "sample_index" in labels_df.columns

train_df = train_df.sort_values(["sample_index", "time"]).reset_index(drop=True)
test_df  = test_df.sort_values(["sample_index", "time"]).reset_index(drop=True)

# =======================
# SECTION 4: Static/dynamic features & text→numeric
# =======================
exclude_cols = {"sample_index", "time"}
feature_cols = [c for c in train_df.columns if c not in exclude_cols]

static_features  = ["n_legs", "n_hands", "n_eyes"]
dynamic_features = [c for c in feature_cols if c not in static_features]

print("Static features:", static_features)
print("Dynamic features count:", len(dynamic_features))

for col in static_features:
    print(col, "unique values:", train_df[col].unique())

text2num = {
    "two": 2,
    "one+peg_leg": 1,
    "one+hook_hand": 1,
    "one+eye_patch": 1,
}

def convert_static_columns(df, static_cols):
    for col in static_cols:
        df[col] = df[col].map(text2num).astype(np.float32)
    return df

train_df = convert_static_columns(train_df, static_features)
test_df  = convert_static_columns(test_df, static_features)

# =======================
# SECTION 5: Build (N, T, C_dyn) & (N, C_static)
# =======================
def df_to_3d_with_static(df, dynamic_cols, static_cols):
    groups = df.groupby("sample_index")

    sample_ids = []
    series_list = []
    static_list = []

    for sid, g in groups:
        g = g.sort_values("time")

        X_dyn = g[dynamic_cols].to_numpy(dtype=np.float32)   # (T, C_dyn)
        X_stat = g[static_cols].iloc[0].to_numpy(dtype=np.float32)

        sample_ids.append(sid)
        series_list.append(X_dyn)
        static_list.append(X_stat)

    series_3d = np.stack(series_list, axis=0)   # (N, T, C_dyn)
    static_2d = np.stack(static_list, axis=0)   # (N, C_static)
    return series_3d, static_2d, np.array(sample_ids)

X_train_dyn, X_train_static, train_ids = df_to_3d_with_static(
    train_df, dynamic_features, static_features
)
X_test_dyn, X_test_static, test_ids = df_to_3d_with_static(
    test_df, dynamic_features, static_features
)

print("Dynamic shape:", X_train_dyn.shape)   # (N, T, C_dyn)
print("Static shape :", X_train_static.shape)

# =======================
# SECTION 6: Label mapping (fixed order)
# =======================
label_order = ["no_pain", "low_pain", "high_pain"]
label2id = {name: idx for idx, name in enumerate(label_order)}
id2label = {idx: name for idx, name in enumerate(label_order)}
print("Label mapping:", label2id)

labels_map = labels_df.set_index("sample_index")["label"].to_dict()
y_train = np.array([label2id[labels_map[sid]] for sid in train_ids], dtype=np.int64)

print("Class counts (full train):", np.bincount(y_train))

# =======================
# SECTION 7: Normalization (dynamic & static)
# =======================
dyn_mean = X_train_dyn.mean(axis=(0, 1), keepdims=True)
dyn_std  = X_train_dyn.std(axis=(0, 1), keepdims=True) + 1e-6
X_train_dyn = (X_train_dyn - dyn_mean) / dyn_std
X_test_dyn  = (X_test_dyn  - dyn_mean) / dyn_std

stat_mean = X_train_static.mean(axis=0, keepdims=True)
stat_std  = X_train_static.std(axis=0, keepdims=True) + 1e-6
X_train_static = (X_train_static - stat_mean) / stat_std
X_test_static  = (X_test_static  - stat_mean) / stat_std

print("Static normalized sample:", X_train_static[:3])

# =======================
# SECTION 8: Outer train/val split
# =======================
skf_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
tr_idx, va_idx = list(skf_outer.split(train_ids, y_train))[0]

X_tr_dyn, X_va_dyn = X_train_dyn[tr_idx], X_train_dyn[va_idx]
X_tr_stat, X_va_stat = X_train_static[tr_idx], X_train_static[va_idx]
y_tr, y_va = y_train[tr_idx], y_train[va_idx]

print("Train split:", X_tr_dyn.shape, "Val split:", X_va_dyn.shape)
print("Train class counts:", np.bincount(y_tr))
print("Val class counts:",   np.bincount(y_va))

# =======================
# SECTION 9: Augmentation
# =======================
def augment_timeseries(x_dyn):
    """
    x_dyn: (T, C_dyn) numpy
    """
    x = x_dyn.copy()

    # jitter
    if np.random.rand() < 0.5:
        noise = np.random.normal(0.0, 0.02, size=x.shape).astype(np.float32)
        x += noise

    # scaling
    if np.random.rand() < 0.5:
        scale = np.random.normal(1.0, 0.05)
        x *= scale

    # small circular shift
    if np.random.rand() < 0.5:
        T = x.shape[0]
        shift = np.random.randint(-5, 6)
        x = np.roll(x, shift, axis=0)

    return x

# =======================
# SECTION 10: Dataset & loaders (oversampling)
# =======================
class TimeSeriesDataset(Dataset):
    def __init__(self, X_dyn, X_stat, y=None, augment=False):
        self.X_dyn = X_dyn
        self.X_stat = X_stat
        self.y = y
        self.augment = augment

    def __len__(self):
        return self.X_dyn.shape[0]

    def __getitem__(self, idx):
        x_dyn = self.X_dyn[idx]      # (T, C_dyn)
        x_stat = self.X_stat[idx]    # (C_static,)

        if self.augment:
            x_dyn = augment_timeseries(x_dyn)

        x_dyn_t = torch.from_numpy(x_dyn).permute(1, 0).contiguous().float()  # (C_dyn, T)
        x_stat_t = torch.from_numpy(x_stat).float()

        if self.y is None:
            return x_dyn_t, x_stat_t

        y_t = torch.tensor(self.y[idx], dtype=torch.long)
        return x_dyn_t, x_stat_t, y_t

def make_weighted_sampler(y_numpy):
    counts = np.bincount(y_numpy)
    class_weights = 1.0 / (counts + 1e-6)
    sample_weights = class_weights[y_numpy]
    sample_weights = torch.from_numpy(sample_weights).float()
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    return sampler

train_ds = TimeSeriesDataset(X_tr_dyn, X_tr_stat, y_tr, augment=True)
val_ds   = TimeSeriesDataset(X_va_dyn, X_va_stat, y_va, augment=False)
test_ds  = TimeSeriesDataset(X_test_dyn, X_test_static, None, augment=False)

train_sampler = make_weighted_sampler(y_tr)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=train_sampler)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

# =======================
# SECTION 11: Model (CNN + BiGRU + Attention + static MLP)
# =======================
class CNNGRUWithAttention(nn.Module):
    def __init__(self, dyn_channels, static_dim, num_classes,
                 gru_hidden=128):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv1d(dyn_channels, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        self.gru = nn.GRU(
            input_size=128,
            hidden_size=gru_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )

        self.attn = nn.Linear(2 * gru_hidden, 1)

        self.static_fc = nn.Sequential(
            nn.Linear(static_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
        )

        self.fc = nn.Linear(2 * gru_hidden + 32, num_classes)

    def forward(self, x_dyn, x_stat):
        # x_dyn: (B, C_dyn, T)
        x = self.cnn(x_dyn)          # (B, 128, T)
        x = x.transpose(1, 2)        # (B, T, 128)

        out_seq, _ = self.gru(x)     # (B, T, 2H)

        attn_scores = self.attn(out_seq).squeeze(-1)   # (B, T)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)  # (B, T, 1)
        context = (out_seq * attn_weights).sum(dim=1)  # (B, 2H)

        x_stat = self.static_fc(x_stat)                # (B, 32)
        h = torch.cat([context, x_stat], dim=1)        # (B, 2H+32)
        logits = self.fc(h)                            # (B, C)
        return logits

dyn_channels = X_train_dyn.shape[2]
static_dim   = X_train_static.shape[1]
num_classes  = len(label_order)

# =======================
# SECTION 12: Focal Loss
# =======================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.0, reduction="mean"):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction
        self.label_smoothing = label_smoothing

        if alpha is not None:
            alpha = torch.tensor(alpha, dtype=torch.float32)
            self.register_buffer("alpha", alpha)
        else:
            self.alpha = None

    def forward(self, logits, target):
        num_classes = logits.size(1)

        log_probs = F.log_softmax(logits, dim=1)
        probs = log_probs.exp()

        with torch.no_grad():
            true_dist = torch.zeros_like(log_probs)
            true_dist.fill_(self.label_smoothing / (num_classes - 1))
            true_dist.scatter_(1, target.unsqueeze(1), 1.0 - self.label_smoothing)

        pt = (probs * true_dist).sum(dim=1)
        focal_factor = (1.0 - pt).pow(self.gamma)

        ce_loss = -(true_dist * log_probs).sum(dim=1)
        loss = focal_factor * ce_loss

        if self.alpha is not None:
            at = self.alpha[target]
            loss = at * loss

        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        else:
            return loss

def make_focal_alpha_from_data(y_numpy, n_classes):
    counts = np.bincount(y_numpy, minlength=n_classes).astype(np.float32)
    freq = counts / counts.sum()
    inv = 1.0 / (freq + 1e-12)
    alpha = inv / inv.sum()
    return alpha.tolist()

def build_criterion(y_numpy, n_classes):
    if USE_FOCAL_LOSS:
        if FOCAL_USE_ALPHA:
            alpha = make_focal_alpha_from_data(y_numpy, n_classes)
            print("Using FocalLoss with alpha:", alpha)
        else:
            alpha = None
            print("Using FocalLoss with NO alpha.")
        return FocalLoss(
            gamma=FOCAL_GAMMA,
            alpha=alpha,
            label_smoothing=FOCAL_LABEL_SMOOTH,
            reduction="mean",
        )
    else:
        print("Using CrossEntropyLoss.")
        return nn.CrossEntropyLoss(label_smoothing=0.1)

# =======================
# SECTION 13: Train / eval helpers
# =======================
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    for x_dyn, x_stat, y in loader:
        x_dyn = x_dyn.to(device)
        x_stat = x_stat.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x_dyn, x_stat)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item() * y.size(0)
        preds = logits.argmax(dim=1)
        all_preds.append(preds.detach().cpu().numpy())
        all_labels.append(y.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, macro_f1

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    for x_dyn, x_stat, y in loader:
        x_dyn = x_dyn.to(device)
        x_stat = x_stat.to(device)
        y = y.to(device)

        logits = model(x_dyn, x_stat)
        loss = criterion(logits, y)
        total_loss += loss.item() * y.size(0)

        preds = logits.argmax(dim=1)
        all_preds.append(preds.cpu().numpy())
        all_labels.append(y.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, macro_f1, all_labels, all_preds

@torch.no_grad()
def predict_proba_any(model, loader, device):
    model.eval()
    all_probs = []
    for batch in loader:
        if len(batch) == 2:
            x_dyn, x_stat = batch
        else:
            x_dyn, x_stat, *_ = batch
        x_dyn = x_dyn.to(device)
        x_stat = x_stat.to(device)
        logits = model(x_dyn, x_stat)
        probs = F.softmax(logits, dim=1)
        all_probs.append(probs.cpu().numpy())
    return np.concatenate(all_probs, axis=0)

def ce_loss_from_probs(probs, y_true):
    eps = 1e-12
    p_true = probs[np.arange(len(y_true)), y_true]
    return -np.log(p_true + eps).mean()

# =======================
# SECTION 14: Main train/val training
# =======================
model = CNNGRUWithAttention(dyn_channels, static_dim, num_classes).to(DEVICE)
criterion = build_criterion(y_tr, num_classes)
optimizer = torch.optim.Adam(model.parameters(), lr=LR_MAIN)

best_val_f1 = -1.0
best_state = None
no_improve = 0

train_losses = []
val_losses = []
train_f1s = []
val_f1s = []

for epoch in range(1, N_EPOCHS_MAIN + 1):
    tr_loss, tr_f1 = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    va_loss, va_f1, _, _ = evaluate(model, val_loader, criterion, DEVICE)

    train_losses.append(tr_loss)
    val_losses.append(va_loss)
    train_f1s.append(tr_f1)
    val_f1s.append(va_f1)

    print(f"Epoch {epoch:02d} | train_loss={tr_loss:.4f}  val_loss={va_loss:.4f}  "
          f"train_F1={tr_f1:.4f}  val_F1={va_f1:.4f}")

    if va_f1 > best_val_f1 + 1e-5:
        best_val_f1 = va_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE_MAIN:
            print("Early stopping triggered.")
            break

if best_state is not None:
    model.load_state_dict(best_state)
torch.save(model.state_dict(), "model_best_val.pt")
print("Saved model_best_val.pt  | best val macro-F1 =", best_val_f1)

# Loss / F1 plots
epochs_range = range(1, len(train_losses)+1)

plt.figure(figsize=(6,4))
plt.plot(epochs_range, train_losses, label="Train Loss (opt step)")
plt.plot(epochs_range, val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Val Loss")
plt.legend()
plt.show()

plt.figure(figsize=(6,4))
plt.plot(epochs_range, train_f1s, label="Train Macro-F1")
plt.plot(epochs_range, val_f1s, label="Val Macro-F1")
plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.title("Train vs Val Macro-F1")
plt.legend()
plt.show()

# =======================
# SECTION 15: Full-train fine-tune on all training data
# =======================
X_full_dyn  = X_train_dyn
X_full_stat = X_train_static
y_full      = y_train

full_ds = TimeSeriesDataset(X_full_dyn, X_full_stat, y_full, augment=True)
full_sampler = make_weighted_sampler(y_full)
full_loader  = DataLoader(full_ds, batch_size=BATCH_SIZE, sampler=full_sampler)

model_full = CNNGRUWithAttention(dyn_channels, static_dim, num_classes).to(DEVICE)
model_full.load_state_dict(torch.load("model_best_val.pt", map_location=DEVICE))

criterion_full = build_criterion(y_full, num_classes)
optimizer_full = torch.optim.Adam(model_full.parameters(), lr=LR_FULL)

for epoch in range(1, FINE_EPOCHS_FULL + 1):
    tr_loss_full, tr_f1_full = train_one_epoch(model_full, full_loader, optimizer_full, criterion_full, DEVICE)
    print(f"[Full] Epoch {epoch:02d} | loss={tr_loss_full:.4f} | F1={tr_f1_full:.4f}")

torch.save(model_full.state_dict(), "model_full_train.pt")
print("Saved model_full_train.pt")

# =======================
# SECTION 16: K-fold CV lite (train K models)
# =======================
def run_kfold_cv_lite(X_dyn_all, X_stat_all, y_all,
                      n_splits=5,
                      n_epochs_cv=N_EPOCHS_CV,
                      patience_cv=PATIENCE_CV,
                      lr_cv=LR_CV):

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    fold_scores = []

    for fold, (tr_idx_cv, va_idx_cv) in enumerate(skf.split(X_dyn_all, y_all), start=1):
        print(f"\n=== K-FOLD {fold}/{n_splits} ===")

        X_tr_cv_dyn, X_va_cv_dyn = X_dyn_all[tr_idx_cv], X_dyn_all[va_idx_cv]
        X_tr_cv_stat, X_va_cv_stat = X_stat_all[tr_idx_cv], X_stat_all[va_idx_cv]
        y_tr_cv, y_va_cv = y_all[tr_idx_cv], y_all[va_idx_cv]

        train_ds_cv = TimeSeriesDataset(X_tr_cv_dyn, X_tr_cv_stat, y_tr_cv, augment=True)
        val_ds_cv   = TimeSeriesDataset(X_va_cv_dyn, X_va_cv_stat, y_va_cv, augment=False)

        sampler_cv = make_weighted_sampler(y_tr_cv)
        train_loader_cv = DataLoader(train_ds_cv, batch_size=BATCH_SIZE, sampler=sampler_cv)
        val_loader_cv   = DataLoader(val_ds_cv,   batch_size=BATCH_SIZE, shuffle=False)

        model_cv = CNNGRUWithAttention(dyn_channels, static_dim, num_classes).to(DEVICE)
        criterion_cv = build_criterion(y_tr_cv, num_classes)
        optimizer_cv = torch.optim.Adam(model_cv.parameters(), lr=lr_cv)

        best_f1_cv = -1.0
        best_state_cv = None
        no_imp_cv = 0

        for epoch in range(1, n_epochs_cv + 1):
            tr_loss_cv, tr_f1_cv = train_one_epoch(model_cv, train_loader_cv, optimizer_cv, criterion_cv, DEVICE)
            va_loss_cv, va_f1_cv, _, _ = evaluate(model_cv, val_loader_cv, criterion_cv, DEVICE)

            print(f"[Fold {fold}] Epoch {epoch:02d} | train_loss={tr_loss_cv:.4f}  "
                  f"val_loss={va_loss_cv:.4f}  train_F1={tr_f1_cv:.4f}  val_F1={va_f1_cv:.4f}")

            if va_f1_cv > best_f1_cv + 1e-5:
                best_f1_cv = va_f1_cv
                best_state_cv = {k: v.cpu().clone() for k, v in model_cv.state_dict().items()}
                no_imp_cv = 0
            else:
                no_imp_cv += 1
                if no_imp_cv >= patience_cv:
                    print(f"[Fold {fold}] Early stopping.")
                    break

        if best_state_cv is not None:
            model_cv.load_state_dict(best_state_cv)
        torch.save(model_cv.state_dict(), f"model_kfold_{fold}.pt")
        print(f"[Fold {fold}] best val macro-F1 = {best_f1_cv:.4f}")
        fold_scores.append(best_f1_cv)

    print("\nK-fold CV scores:", fold_scores)
    print("Mean F1:", np.mean(fold_scores))

run_kfold_cv_lite(X_train_dyn, X_train_static, y_train)

# =======================
# SECTION 17: Inference on TEST + ensembles + VAL metrics
# =======================
print("\n=== Inference on TEST + VAL metrics ===\n")

y_va_np = y_va.copy()

# --- best-val model ---
print("--- Best-val model ---")
model_best = CNNGRUWithAttention(dyn_channels, static_dim, num_classes).to(DEVICE)
model_best.load_state_dict(torch.load("model_best_val.pt", map_location=DEVICE))
criterion_eval = build_criterion(y_tr, num_classes)

val_loss_best, val_f1_best, y_va_best, preds_va_best = evaluate(model_best, val_loader, criterion_eval, DEVICE)
print(f"[VAL] best-val model:   loss={val_loss_best:.4f}, macroF1={val_f1_best:.4f}")

val_probs_best  = predict_proba_any(model_best, val_loader, DEVICE)
test_probs_best = predict_proba_any(model_best, test_loader, DEVICE)

test_pred_ids_best    = test_probs_best.argmax(axis=1)
test_pred_labels_best = [id2label[i] for i in test_pred_ids_best]
submission_best = pd.DataFrame({
    "sample_index": test_ids,
    "label": test_pred_labels_best
}).sort_values("sample_index")
submission_best.to_csv("submission_best_val.csv", index=False)
print("Saved submission_best_val.csv")

# --- full-train model ---
print("\n--- Full-train model ---")
model_full_load = CNNGRUWithAttention(dyn_channels, static_dim, num_classes).to(DEVICE)
model_full_load.load_state_dict(torch.load("model_full_train.pt", map_location=DEVICE))

val_loss_full, val_f1_full, y_va_full, preds_va_full = evaluate(model_full_load, val_loader, criterion_eval, DEVICE)
print(f"[VAL] full-train model: loss={val_loss_full:.4f}, macroF1={val_f1_full:.4f}")

val_probs_full  = predict_proba_any(model_full_load, val_loader, DEVICE)
test_probs_full = predict_proba_any(model_full_load, test_loader, DEVICE)

test_pred_ids_full    = test_probs_full.argmax(axis=1)
test_pred_labels_full = [id2label[i] for i in test_pred_ids_full]
submission_full = pd.DataFrame({
    "sample_index": test_ids,
    "label": test_pred_labels_full
}).sort_values("sample_index")
submission_full.to_csv("submission_full_train.csv", index=False)
print("Saved submission_full_train.csv")

# --- 2-model ensemble ---
print("\n--- Ensemble: best-val + full-train ---")
val_probs_2ens  = 0.5 * val_probs_best  + 0.5 * val_probs_full
test_probs_2ens = 0.5 * test_probs_best + 0.5 * test_probs_full

val_preds_2ens = val_probs_2ens.argmax(axis=1)
val_f1_2ens    = f1_score(y_va_np, val_preds_2ens, average="macro")
val_loss_2ens  = ce_loss_from_probs(val_probs_2ens, y_va_np)
print(f"[VAL] 2-model ensemble: loss={val_loss_2ens:.4f}, macroF1={val_f1_2ens:.4f}")

test_pred_ids_2ens    = test_probs_2ens.argmax(axis=1)
test_pred_labels_2ens = [id2label[i] for i in test_pred_ids_2ens]
submission_2ens = pd.DataFrame({
    "sample_index": test_ids,
    "label": test_pred_labels_2ens
}).sort_values("sample_index")
submission_2ens.to_csv("submission_ensemble_2models.csv", index=False)
print("Saved submission_ensemble_2models.csv")

# --- K-fold ensemble ---
print("\n--- K-fold ensemble (models from run_kfold_cv_lite) ---")
kfold_probs_val_list  = []
kfold_probs_test_list = []
n_splits_k = 5

for fold in range(1, n_splits_k + 1):
    path_fold = f"model_kfold_{fold}.pt"
    if not os.path.exists(path_fold):
        print(f"WARNING: {path_fold} not found, skipping this fold.")
        continue

    m_fold = CNNGRUWithAttention(dyn_channels, static_dim, num_classes).to(DEVICE)
    m_fold.load_state_dict(torch.load(path_fold, map_location=DEVICE))

    probs_val_fold  = predict_proba_any(m_fold, val_loader, DEVICE)
    probs_test_fold = predict_proba_any(m_fold, test_loader, DEVICE)

    kfold_probs_val_list.append(probs_val_fold)
    kfold_probs_test_list.append(probs_test_fold)

val_probs_kfold_ens = None
test_probs_kfold_ens = None
val_preds_kfold = None
val_f1_kfold = None
val_loss_kfold = None

if len(kfold_probs_val_list) > 0:
    kfold_probs_val_stack  = np.stack(kfold_probs_val_list,  axis=0)
    kfold_probs_test_stack = np.stack(kfold_probs_test_list, axis=0)

    val_probs_kfold_ens  = kfold_probs_val_stack.mean(axis=0)
    test_probs_kfold_ens = kfold_probs_test_stack.mean(axis=0)

    val_preds_kfold = val_probs_kfold_ens.argmax(axis=1)
    val_f1_kfold    = f1_score(y_va_np, val_preds_kfold, average="macro")
    val_loss_kfold  = ce_loss_from_probs(val_probs_kfold_ens, y_va_np)
    print(f"[VAL] K-fold ensemble:  loss={val_loss_kfold:.4f}, macroF1={val_f1_kfold:.4f}")

    test_pred_ids_kens    = test_probs_kfold_ens.argmax(axis=1)
    test_pred_labels_kens = [id2label[i] for i in test_pred_ids_kens]
    submission_kfold_ens = pd.DataFrame({
        "sample_index": test_ids,
        "label": test_pred_labels_kens
    }).sort_values("sample_index")
    submission_kfold_ens.to_csv("submission_kfold_ensemble.csv", index=False)
    print("Saved submission_kfold_ensemble.csv")
else:
    print("No k-fold models found; skipping K-fold ensemble.")

# --- FINAL ensemble: 2-model + K-fold ---
val_preds_final = None
val_f1_final = None
val_loss_final = None

if val_probs_kfold_ens is not None:
    print("\n--- FINAL ensemble: (2-model ensemble) + (K-fold ensemble) ---")
    val_probs_final  = 0.5 * val_probs_2ens  + 0.5 * val_probs_kfold_ens
    test_probs_final = 0.5 * test_probs_2ens + 0.5 * test_probs_kfold_ens

    val_preds_final = val_probs_final.argmax(axis=1)
    val_f1_final    = f1_score(y_va_np, val_preds_final, average="macro")
    val_loss_final  = ce_loss_from_probs(val_probs_final, y_va_np)
    print(f"[VAL] FINAL ensemble:  loss={val_loss_final:.4f}, macroF1={val_f1_final:.4f}")

    test_pred_ids_final    = test_probs_final.argmax(axis=1)
    test_pred_labels_final = [id2label[i] for i in test_pred_ids_final]
    submission_final = pd.DataFrame({
        "sample_index": test_ids,
        "label": test_pred_labels_final
    }).sort_values("sample_index")
    submission_final.to_csv("submission_final_ensemble.csv", index=False)
    print("Saved submission_final_ensemble.csv")
else:
    print("No K-fold ensemble; skipping FINAL ensemble.")

# =======================
# SECTION 18: Confusion matrices on VAL for all final models
# =======================
def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=[0,1,2])
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[id2label[i] for i in [0,1,2]]
    )
    disp.plot(values_format="d", cmap="Blues", colorbar=False)
    plt.title(title)
    plt.show()

print("\n=== Confusion matrices on VAL for all final models ===")
plot_cm(y_va_best, preds_va_best, "Confusion Matrix - VAL (best-val model)")
plot_cm(y_va_full, preds_va_full, "Confusion Matrix - VAL (full-train model)")
plot_cm(y_va_np, val_preds_2ens, "Confusion Matrix - VAL (2-model ensemble)")

if val_preds_kfold is not None:
    plot_cm(y_va_np, val_preds_kfold, "Confusion Matrix - VAL (K-fold ensemble)")

if val_preds_final is not None:
    plot_cm(y_va_np, val_preds_final, "Confusion Matrix - VAL (FINAL ensemble)")
